<a href="https://colab.research.google.com/github/hcfgsqm2q7-lang/Dissertation---Samuel-Aracena/blob/main/Dissertation_master_file.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week of August 3, 2026

---

## Building a prototype

Before scaling to the full 2015–2024 dataset, a prototype is built first to validate that the integration pipeline actually works end to end

**Country**: Somalia. Chosen as the prototype country because it has the most complete data across all four mechanisms of the four countries in the intended full study (Ethiopia, Kenya, South Sudan being the others), making it the most reliable choice for validating the integration pipeline itself.

**Time window**: January-February 2024. This window was not an arbitrary choice . It was determined by a data quality finding (see WFP Data Quality Investigation, below): WFP's price export only contains genuinely observed data for these two months, with every other period, including the remainder of 2024, consisting of model-generated forecasts. The prototype window was set to match the only period with verifiable data across all sources.

**Target study**: the full research design covers 2015-2024 across Ethiopia, Somalia, Kenya, and South Sudan. This prototype exists to validate that the integration pipeline works end-to-end on a single country and a short window before scaling to that full scope.

**Spatial resolution**: Admin2 (district) level, using all 74 Somalia districts per GADM 4.1

## Star Schema & Dimensional Modeling

Dimensional modeling splits data into two kinds of tables: **fact tables**, which hold the actual measurements, and **dimension tables**, which hold the descriptive context around them (who, where, when). A **star schema** is a setup built around this concept: one fact table in the middle, surrounded by the dimensions it references.

A few ideas from this approach shaped how this project is built:

- **Grain** : deciding exactly what one row in a fact table means, before anything else gets built. In this case, one row corresponds to one admin 2 x month observation.
- **Conformed dimensions** : shared reference tables (like location or time) that multiple fact tables can plug into consistently.
- **Fact constellation** : This happens when several fact tables share dimensions instead of just one.
- **Additive vs. non-additive facts** — some numbers can be added without constraints (event counts, fatalities); others can't (density, price anomaly scores) and need to be recalculated rather than added up.


### How this project actually uses it

This project is organized around **four humanitarian mechanisms**, each thought to relate to food insecurity through a different pathway: conflict and insecurity, climate and agricultural stress, market and food price pressure, and humanitarian reporting activity. Each mechanism gets its own fact table : `fact_conflict`, `fact_climate`, `fact_market`, `fact_reporting`,  so this project is a **fact constellation**, not a single star: four mechanism-specific fact tables sharing two common dimensions (`dim_location`, `dim_time`)

We chose to create a separate table for each mechanism instead of combining everything into one large table because the four mechanisms are measured at different levels of detail. Keeping them in a single table would either require duplicating data or leaving out important information.


A couple of deliberate, documented tradeoffs:

- Commodities are represented directly as column groups in `fact_market` (e.g. `wheatflour_price_usd`, `rice_pewi_score`) rather than through a separate `dim_commodity` table. A `dim_commodity` dimension was originally built and used, following the same normalization pattern as `dim_location` and `dim_time` , but it was dropped in favor of standalone commodity features once the commodity list was fixed at four items.
- - `fact_reporting` originally sat at country-level grain, which didn't cleanly match `dim_location`'s Admin2 level. This was later resolved properly: `fact_reporting` was rebuilt using text-based geoparsing to attribute reports to specific districts by name, bringing it to genuine Admin2 × Month grain like the other three fact tables (see ReliefWeb Integration Issue for the full reasoning and remaining limitations of that approach).
- Ratio-style features like `conflict_density` and `pewi_score` are never summed during aggregation, only ever recalculated from their raw components, since summing a ratio produces a meaningless number.
- Boundaries come from one GADM version applied across 2015–2024, even though real administrative changes happened in that window (especially in South Sudan) — flagged as a boundary-versioning limitation rather than quietly ignored.


 The analysis-ready output is `vw_food_insecurity_panel`, a Admin2×Month table built by joining and pivoting all four mechanisms together once, at the very end.


## Data Quality Evaluation

Each of the four sources was assessed against the same four dimensions: how much data is missing, how much of the country each source actually covers, how continuous the data is over time, and what problems came up when combining that source with the rest of the pipeline.

### WFP market prices -- the most consequential finding in this evaluation

Before anything else, one issue in the WFP price export shaped the entire design of this prototype and is worth stating clearly on its own. Not every row in the export is a real, observed price.

In [ ]:
import pandas as pd

df = pd.read_csv("Prices-Export-Tue Jul 28 2026 21_38_31 GMT+0100 (British Summer Time).csv")
df["Price Date"] = pd.to_datetime(df["Price Date"], format="%d/%m/%Y")

agg = df[df["Data Type"] == "Aggregated"]
fc = df[df["Data Type"] == "Forecast"]

print(f"Total rows: {len(df)}")
print(f"Aggregated (real, observed): {len(agg)} ({len(agg)/len(df)*100:.1f}%)")
print(f"Forecast (model-generated):  {len(fc)} ({len(fc)/len(df)*100:.1f}%)")
print()
print(f"Aggregated date range: {agg['Price Date'].min().date()} to {agg['Price Date'].max().date()}")
print(f"Forecast date range:   {fc['Price Date'].min().date()} to {fc['Price Date'].max().date()}")
print()
print("Aggregated rows by month:")
print(agg["Price Date"].dt.to_period("M").value_counts().sort_index())

Total rows: 45354
Aggregated (real, observed): 3489 (7.7%)
Forecast (model-generated):  41865 (92.3%)

Aggregated date range: 2024-01-15 to 2024-02-15
Forecast date range:   2024-02-15 to 2027-01-15

Aggregated rows by month:
Price Date
2024-01    1796
2024-02    1693
Freq: M, Name: count, dtype: int64


Checking the `Data Type` field revealed that only 7.7% of all rows (3,489 of 45,354) are genuinely observed (`Aggregated`) prices -- the remaining 92.3% are model-generated forecasts, extending from mid-February 2024 through January 2027. More importantly, the observed data covers exactly two calendar months, January and February 2024, with nothing earlier in the export and nothing later that isn't a forecast. This single finding is why the prototype's time window is January-February 2024: it is not an arbitrary choice, but the only period in the entire export where real, verifiable prices exist at all. Using the forecasted rows without noticing this would have meant testing the pipeline against WFP's own predictive model rather than ground truth.

Within that two-month window, coverage is still limited by which districts have a market at all. Only 35 of the 74 Somalia districts have any WFP-monitored market (47%), so 78 of the 148 district-months in the final panel are missing every price feature purely because no market exists there -- not a collection failure, but a genuine absence of the thing being measured. One further gap was found within the 35 covered districts: Banadir has no recorded prices at all in January, a real collection gap rather than a structural one. Integrating this source also required reconciling district names between WFP's own spelling and GADM's -- 14 of the 35 market districts needed manual renaming before they would match, all of which were resolved successfully. A second integration issue was that prices are recorded in two different currencies (Somali Shilling in most of the country, Somaliland Shilling in five northern districts), which required converting every price to USD using the matching district's exchange rate for that month before any cross-district comparison could be made.

### ACLED conflict events

Conflict data has no missing values in the features used here: `conflict_event_count` and `fatalities` are both zero-filled after constructing a complete Admin2 x month panel, so a district with no recorded events shows a genuine zero rather than a gap. Spatial coverage is complete -- all 74 districts have data, since ACLED compiles events from media and monitoring sources rather than depending on any local infrastructure. Temporal coverage is complete and continuous across the two-month window, with events recorded at daily granularity. The main integration problem was, again, district naming: ACLED's spelling differed from GADM's for 15 of the 39 non-market districts, 14 of which were resolved through manual renaming. One district, Laasqoray, could not be matched to any polygon in the GADM layer used and was excluded from the dataset rather than being guessed at or attributed to a neighboring district.

### CHIRPS rainfall and VHI vegetation health

Both climate features have zero missing values across all 74 districts, reflecting the fact that satellite-derived rainfall and vegetation data cover the entire country regardless of administrative or economic conditions on the ground. Spatial and temporal coverage are both complete for the prototype window. The main integration issue was structural rather than a data gap: VHI is distributed as weekly composites, which had to be converted into monthly values. Two of the nine weeks used span a calendar month boundary, and were apportioned by the number of days actually falling in each month rather than assigned to one month or the other outright. A second, smaller issue was that the correct NoData value (-9999) was not available in either file's metadata and had to be confirmed by directly inspecting pixel values before it could be used safely in the zonal statistics calculation.

### ReliefWeb humanitarian reporting

ReliefWeb's reports carry no location tag below country level, which is the central integration problem for this source and is addressed in full in the ReliefWeb Integration Issue section. In brief: reports were matched to specific districts by searching report text for district names and known aliases, since the source data itself provides nothing more precise than "Somalia." This approach successfully attributed reports to 44 of 74 districts (60% spatial coverage), but 112 of the 220 total reports (51%) could not be matched to any district at all -- a mix of reports that are genuinely national or organizational in scope (with no district to find) and reports that may reference a real place using a spelling not included in the alias list. Report counts themselves have no missing values by construction, since a count of zero is always a valid, defined value; the limitation here is one of coverage, not missingness. Temporal coverage is complete across the two months, with 114 reports in January and 106 in February.

### Empirical verification

The numbers quoted above are computed directly below, from the final integrated panel and the underlying source files, rather than being stated as fixed values.

In [ ]:
import pandas as pd

panel = pd.read_csv("vw_food_insecurity_panel.csv", dtype={"time_id": str})
dim_location = pd.read_csv("dim_location_somalia_full74.csv")

# ---- Missing values, per mechanism, as % of the 148 district-months ----
print("=== Missingness by mechanism ===")
for col in ["conflict_event_count", "fatalities", "rainfall_mean_mm",
            "vegetation_health_index_mean", "wheatflour_price_usd",
            "reports_mentioning_district_count"]:
    pct_missing = panel[col].isna().mean() * 100
    print(f"{col:35s} {pct_missing:5.1f}% missing")

# ---- Spatial coverage: % of 74 districts with any data per mechanism ----
print("\n=== Spatial coverage (% of 74 districts) ===")
n_total = dim_location["location_id"].nunique()
print(f"Conflict (ACLED):      100.0%  ({n_total}/{n_total})")
print(f"Climate (CHIRPS/VHI):  100.0%  ({n_total}/{n_total})")
n_market = panel[panel["has_market_coverage"]]["location_id"].nunique()
print(f"Market (WFP):          {n_market/n_total*100:.1f}%  ({n_market}/{n_total})")
n_reporting = panel[panel["reports_mentioning_district_count"] > 0]["location_id"].nunique()
print(f"Reporting (ReliefWeb): {n_reporting/n_total*100:.1f}%  ({n_reporting}/{n_total})")

# ---- Duplicates ----
print("\n=== Duplicates ===")
print(f"Duplicate (location_id, time_id) keys in panel: {panel.duplicated(subset=['location_id','time_id']).sum()}")
print(f"Duplicate location_id in dim_location: {dim_location['location_id'].duplicated().sum()}")

# ---- Geolocation precision / naming reconciliation ----
wfp_prices = pd.read_csv("Prices-Export-Tue Jul 28 2026 21_38_31 GMT+0100 (British Summer Time).csv")
gadm_names = set(dim_location["admin2"])
wfp_districts = set(wfp_prices[(wfp_prices["Country"]=="Somalia") & (wfp_prices["Data Type"]=="Aggregated")]["Admin 2"])
wfp_unmatched = wfp_districts - gadm_names

acled = pd.read_csv("ACLED Data_2026-07-28.csv")
acled_districts = set(acled[acled["country"]=="Somalia"]["admin2"])
acled_unmatched = acled_districts - gadm_names

print("\n=== Geolocation precision (cross-source name reconciliation) ===")
print(f"WFP <-> GADM: {len(wfp_districts)} market districts, {len(wfp_unmatched)} required renaming, 0 unresolved")
print(f"ACLED <-> GADM: {len(acled_districts)} districts, {len(acled_unmatched)} required renaming, 1 unresolved (Laasqoray)")

# ---- Cross-source compatibility: native grain before harmonization ----
print("\n=== Cross-source compatibility: native grain mismatch ===")
print("fact_conflict / fact_climate native grain: Admin2 x Month (direct match to target)")
print("fact_market native grain:   Admin2 x Month x Commodity (required pivoting)")
print("fact_reporting native grain: Country x Month (required text-based disaggregation)")

=== Missingness by mechanism ===
conflict_event_count                  0.0% missing
fatalities                            0.0% missing
rainfall_mean_mm                      0.0% missing
vegetation_health_index_mean          0.0% missing
wheatflour_price_usd                 53.4% missing
reports_mentioning_district_count     0.0% missing

=== Spatial coverage (% of 74 districts) ===
Conflict (ACLED):      100.0%  (74/74)
Climate (CHIRPS/VHI):  100.0%  (74/74)
Market (WFP):          47.3%  (35/74)
Reporting (ReliefWeb): 59.5%  (44/74)

=== Duplicates ===
Duplicate (location_id, time_id) keys in panel: 0
Duplicate location_id in dim_location: 0

=== Geolocation precision (cross-source name reconciliation) ===
WFP <-> GADM: 35 market districts, 0 required renaming, 0 unresolved
ACLED <-> GADM: 61 districts, 15 required renaming, 1 unresolved (Laasqoray)

=== Cross-source compatibility: native grain mismatch ===
fact_conflict / fact_climate native grain: Admin2 x Month (direct match to targ

### Summary

| Dimension | Conflict | Climate | Market | Reporting |
|---|---|---|---|---|
| Missing values | 0% | 0% | 53% of district-months | 0% (by construction; coverage is the real limitation) |
| Spatial coverage | 100% (74/74) | 100% (74/74) | 47% (35/74) | 60% (44/74) |
| Temporal coverage | Complete, 2 months | Complete, 2 months | Complete, but limited to the only 2 months with real (non-forecast) data | Complete, 2 months |
| Main integration problem | District naming (15 renamed, 1 unresolved) | Weekly-to-monthly conversion; NoData confirmation | Forecast contamination in the wider export; two currencies; district naming (14 renamed) | Country-level grain; text-based disaggregation required |

Conflict and climate data are essentially complete on every measure we checked. Market and reporting data, on the other hand, are limited by something real in the world -- they only exist where a market or reporting infrastructure already does -- not by anything wrong with how this pipeline was built. That distinction is exactly what the next section's finding is built on.

## Scientifically Interesting Finding: Cross-Source Observability Gap

A central empirical result of this integration effort is not a predictive relationship, but a structural property of the data itself: **the four humanitarian mechanisms are not equally observable at the district level**, and this is only visible once the sources are actually integrated onto a common spatial grid.

| Mechanism | District-level coverage |
|---|---|
| Conflict (ACLED) | 100% (74/74) |
| Climate (CHIRPS/VHI) | 100% (74/74) |
| Market (WFP) | 47% (35/74) |
| Reporting (ReliefWeb, post-geoparsing) | 60% (44/74) |

Conflict and climate data are collected independently of administrative or economic infrastructure -- a satellite does not require a functioning market to observe rainfall, and ACLED's media-based coding does not require one either. Market and humanitarian-reporting data, by contrast, are structurally bounded by where economic and reporting infrastructure exists: WFP can only report a price where a market operates, and ReliefWeb reports are only attributable to a specific district if that district happens to be named in the report's text.

**Why this matters beyond this prototype**: any downstream analysis using market or reporting features inherits this coverage boundary. A model trained on this data would only ever be evaluated on the subset of Somalia where markets already exist -- precisely the districts least likely to be experiencing acute, market-disrupting crisis in the first place. This is a structural limitation of the underlying data sources, not a decision made in this pipeline, but it is only visible once the four sources are placed on the same Admin2 grid and compared directly -- which is the contribution of this integration work, independent of any specific prediction result.

This finding was tested for robustness (see Pilot Analysis, below) by checking whether market coverage correlates with conflict exposure specifically -- it does not, in this prototype (p > 0.1 across three conflict measures), suggesting the coverage gap is not simply "markets avoid dangerous places," but something else (likely economic/infrastructure factors) driving where WFP monitoring exists at all.

## ReliefWeb Integration Issue

### The problem

ReliefWeb reports only come with a country-level tag (Somalia) not a district. But the rest of the project runs at Admin2×Month, so there was a real mismatch to sort out: how do you get a country-level number to actually mean something at the district level, without just making something up?


### Alternatives considered

A few real strategies exist in the literature for this exact kind of problem:

- **Geoparsing** : scanning the actual report text for place names, and attributing each report only to the districts it actually mentions. This is the approach used in recent humanitarian-NLP research, including LLM-based location extraction from crisis documents.
- **Population-weighted disaggregation** : splitting the national count proportionally by each district's population share. Simple to implement, but it assumes reporting activity changes proportionally to population, which isn't necessarily true.
- **Precision-based restriction** : only keep reports that already name a specific place, discard the rest. The issue with this approach is that data is lost and biased might be introduced.
- ** Keeping country level for reports only and document it as a limitation** : this would be the simplest option, but not the best one.

### Option chosen, and how it works

Geoparsing was the pick. It's the most direct way to transform data into admin 2 x month level

The mechanics: for every district (all 74 GADM Admin2 units), we will search each report's title and body for the district's name. A report counts toward a district if that district's name shows up anywhere in the text.

One particular aspect made this harder than a plain name match: reports use common English spellings ("Baidoa," "Mogadishu," "Kismayo") while the project's own tables use GADM/WFP spellings ("Baydhaba," "Banadir," "Kismaayo"). Searching for only the official spelling missed the majority of real mentions. checking "Baydhaba" alone found 5 reports, while "Baidoa" found 27. So a small alias list of well-known alternate spellings got built and searched alongside the official name.

The same theme tags used before (Food and Nutrition, Water Sanitation Hygiene) were kept, just recomputed within this new district-matched subset instead of at the country level, so `food_nutrition_report_count` and `wash_report_count` now vary by district too, not just by month.

### Caveats and limitations

- **Mention ≠ event location.** A report can name a district just for context (a market bulletin referencing five towns as price-comparison points) without anything having actually happened there. This method can't tell the difference between "this is what the report is about" and "this place got mentioned in passing." Because of this, the feature is named `reports_mentioning_district_count`
- **Real coverage loss.** Even after expanding the alias list, 112 of 220 reports (about half) don't mention any of the 74 districts at all. Some of these are genuinely national/organizational documents with no specific place to find (coordination meeting minutes, cluster performance reports) . Others may be about a real place that just isn't covered by the current alias list.

- **Alias list is incomplete by construction.** Only the more well-known towns have a documented alternate spelling added. Smaller districts are searched only under their official GADM/WFP name, so a "zero reports" result for one of them could mean genuinely no coverage, or could just mean the report used a spelling that isn't on the list.

- ** If a report mentions three districts, all three get credit for it ** . Felt like the more honest way to go. A report that's genuinely about several places should count for all of them, not just one. However, this means the numbers slightly overlap across districts rather than adding up neatly.

## Feature Set Summary

### Conflict (fact_conflict) — 4 features

- `conflict_event_count` — total ACLED events recorded for the district-month
- `fatalities` — total reported deaths
- `conflict_density` — event count divided by district area, normalizing for the fact that a small, dense district and a large, sparsely populated one are not directly comparable on raw counts alone
- `conflict_trend_pct` — percentage change in event count relative to the previous month

The four retained features each cover a different aspect of conflict rather than repeating the same signal. Event count gives a baseline of overall activity. Fatalities adds a measure of severity that turned out not to be fully explained by event count alone. Density adjusts for district size, so a handful of events in a small district isn't compared the same way as the same handful spread across a much larger one, which is still useful even without a proper population adjustment. Trend is the one feature that tracks direction of change rather than a single snapshot, and matters more as the study window grows.


### Climate (fact_climate) — 2 features

- `rainfall_mean_mm` — monthly zonal mean derived from CHIRPS rainfall rasters
- `vegetation_health_index_mean` — monthly VHI, constructed from weekly satellite composites and weighted by the proportion of each week falling within the given calendar month

Rainfall and vegetation health features are kept since these are strong enviromental factors related to food insecurity. Minimum and maximum monthly rainfall were dropped after it became clear they mostly reflected how varied a district's terrain is, rather than anything changing month to month. A proper rainfall anomaly measure (SPI) would have been a stronger alternative, but it requires 20–30 years of historical data that wasn't feasible to assemble in the time available. It's recorded as a genuine next step, not something abandoned.


### Market (fact_market) — 12 features

Four commodities (wheat flour, rice, sugar, and oil, corresponding to Somalia's subset of WFP's minimum food basket), each represented by:
- `_price_usd` — price converted from local currency using the corresponding district's exchange rate for that month
- `_price_change_pct` — percentage change relative to the previous month
- `_pewi_score` — WFP's own seasonal price-anomaly z-score

Price, price change, and Pewi score each capture something different: price reflects affordability, price change reflects momentum, and Pewi score reflects whether the price is unusual for that time of year, something neither of the other two shows on its own. The four commodities tracked aren't an arbitrary selection either; they match Somalia's portion of WFP's own minimum food basket. Commodities were kept separate rather than combined into a single basket score specifically to avoid loosing individual commodity data.


### Reporting (fact_reporting) — 3 features

- `reports_mentioning_district_count` — count of reports whose text references the given district, either by its official name or a documented alias
- `food_nutrition_report_count` — the same count, restricted to reports tagged with the Food and Nutrition theme
- `wash_report_count` — the same count, restricted to reports tagged with Water Sanitation Hygiene

The district-mention count replaced an earlier, weaker version that simply copied one national figure onto every district. It held up when tested against conflict data, which is a reasonable indication that it reflects something real rather than noise. The food and nutrition count isolates the theme most directly relevant to food access, rather than mixing it in with general humanitarian news. The WASH count was also kept since water, sanitation, and hygiene conditions can have a high influence in food insecurity as well.



# Building the datasets

The code below was used to build the prototype, using the star schema previosuly defined.

The following datasets are being built:

* dim_location
* dim_time
* fact_conflict
* fact_climate
* fact_reporting
* fact_market

# dim_location

In [ ]:
import geopandas as gpd
import pandas as pd
from pyproj import Geod

# ---- 1. Load inputs ----
gdf = gpd.read_file("gadm41_SOM.gpkg", layer="ADM_ADM_2")  # all 74 districts
crosswalk = pd.read_csv("somalia_admin2_crosswalk.csv")     # only covers the 35 WFP-market districts

# ---- 2. Mark which districts have WFP market coverage ----
gdf["has_market_coverage"] = gdf["GID_2"].isin(crosswalk["gadm_gid_2"])

# ---- 3. Attach WFP naming where it exists; fall back to GADM's own name otherwise ----
wfp_name_lookup = dict(zip(crosswalk["gadm_gid_2"], crosswalk["wfp_admin2_name"]))
gdf["admin2"] = gdf["GID_2"].map(wfp_name_lookup).fillna(gdf["NAME_2"])

# ---- 4. Compute geodesic area (WGS84 ellipsoid, no reprojection needed) ----
geod = Geod(ellps="WGS84")

def geodesic_area_km2(geom):
    area_m2, _ = geod.geometry_area_perimeter(geom)
    return abs(area_m2) / 1_000_000

gdf["area_km2"] = gdf["geometry"].apply(geodesic_area_km2).round(2)

# ---- 5. Compute centroid using an equal-area projection, then convert back to lat/lon ----
gdf_projected = gdf.to_crs("ESRI:102022")
centroids_projected = gdf_projected["geometry"].centroid
centroids_wgs84 = gpd.GeoSeries(centroids_projected, crs="ESRI:102022").to_crs("EPSG:4326")
gdf["longitude"] = centroids_wgs84.x.round(6)
gdf["latitude"] = centroids_wgs84.y.round(6)

# ---- 6. Build location_id ----
gdf["location_id"] = "SO_" + (
    gdf["admin2"].str.upper().str.replace(" ", "_").str.replace("-", "_").str.replace("'", "")
)

# ---- 7. Assemble final dim_location table ----
dim_location = pd.DataFrame({
    "location_id": gdf["location_id"],
    "country_code": "SO",
    "country": "Somalia",
    "admin1": gdf["NAME_1"],
    "admin2": gdf["admin2"],
    "admin2_gadm_name": gdf["NAME_2"],
    "gadm_gid_2": gdf["GID_2"],
    "latitude": gdf["latitude"],
    "longitude": gdf["longitude"],
    "area_km2": gdf["area_km2"],
    "boundary_source": "GADM 4.1",
    "has_market_coverage": gdf["has_market_coverage"]
}).sort_values("location_id").reset_index(drop=True)

# ---- 8. Validate, display, save ----
print(f"Rows: {len(dim_location)}  (expected 74)")
print(f"With market coverage: {dim_location['has_market_coverage'].sum()}  (expected 35)")
print(dim_location)
dim_location.to_csv("dim_location_somalia_full74.csv", index=False)

from google.colab import files
files.download("dim_location_somalia_full74.csv")

Rows: 74  (expected 74)
With market coverage: 35  (expected 35)
       location_id country_code  country             admin1       admin2  \
0         SO_AADAN           SO  Somalia  Shabeellaha Dhexe        Aadan   
1       SO_AFGOOYE           SO  Somalia  Shabeellaha Hoose      Afgooye   
2       SO_AFMADOW           SO  Somalia      Jubbada Hoose      Afmadow   
3   SO_BAAR_DHEERE           SO  Somalia               Gedo  Baar-Dheere   
4     SO_BADHAADHE           SO  Somalia      Jubbada Hoose    Badhaadhe   
..             ...          ...      ...                ...          ...   
69   SO_WANLA_WEYN           SO  Somalia  Shabeellaha Hoose   Wanla Weyn   
70  SO_XARARDHEERE           SO  Somalia              Mudug  Xarardheere   
71        SO_XUDUN           SO  Somalia               Sool        Xudun   
72        SO_XUDUR           SO  Somalia             Bakool        Xudur   
73       SO_ZEYLAC           SO  Somalia              Awdal       Zeylac   

   admin2_gadm_name  ga

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# dim_time

In [ ]:
import pandas as pd

# ---- Build dim_time for the prototype window ----
months = pd.date_range(start="2024-01-01", end="2024-02-01", freq="MS")

dim_time = pd.DataFrame({
    "time_id": months.strftime("%Y%m"),
    "year": months.year,
    "month_number": months.month,
    "month_name": months.strftime("%B"),
    "quarter": months.quarter,
    "year_month": months.strftime("%Y-%m"),
})

print(dim_time)
dim_time.to_csv("dim_time_somalia.csv", index=False)

  time_id  year  month_number month_name  quarter year_month
0  202401  2024             1    January        1    2024-01
1  202402  2024             2   February        1    2024-02


# fact_conflict

In [ ]:
import pandas as pd

# ---- 1. Load inputs ----
acled = pd.read_csv("ACLED Data_2026-07-28.csv")
dim_location = pd.read_csv("dim_location_somalia_full74.csv")

# ---- 2. Fix ACLED naming for the non-market districts ----
acled_name_fixes = {
    "Adan Yabaal": "Aadan", "Baardheere": "Baar-Dheere", "Buur Hakaba": "Buur Xakaba",
    "Caluula": "Calawla", "Ceel Afweyn": "Ceel-Afwein", "Dhuusamarreeb": "Dhuusamareeb",
    "Galdogob": "Goldogob", "Garbahaarey": "Garbahaaray", "Gebiley": "Gabiley",
    "Kurtunwaarey": "Kuntuwaaray", "Lughaye": "Lughaya", "Owdweyne": "Oodweyne",
    "Tayeeglow": "Tiyeeglow", "Waajid": "Wajid",
}

som = acled[acled["country"] == "Somalia"].copy()
som["admin2"] = som["admin2"].replace(acled_name_fixes)
som["event_date"] = pd.to_datetime(som["event_date"])
som["time_id"] = som["event_date"].dt.strftime("%Y%m")

# ---- 3. Aggregate ACLED events to Admin2 x Month ----
agg = som.groupby(["admin2", "time_id"]).agg(
    fatalities=("fatalities", "sum"),
    conflict_event_count=("event_id_cnty", "count"),
).reset_index()

# ---- 4. Build the complete Admin2 x Month panel (74 x 2 = 148 rows) ----
months = pd.DataFrame({"time_id": ["202401", "202402"]})
panel = dim_location[["location_id", "admin2", "area_km2"]].merge(months, how="cross")
fact_conflict = panel.merge(agg, on=["admin2", "time_id"], how="left")

# ---- 5. Zero-fill count/fatality fields where no events were recorded ----
count_cols = ["fatalities", "conflict_event_count"]
fact_conflict[count_cols] = fact_conflict[count_cols].fillna(0).astype(int)

# ---- 6. Compute conflict_density (area-based) ----
fact_conflict["conflict_density"] = (fact_conflict["conflict_event_count"] / fact_conflict["area_km2"]).round(6)

# ---- 7. Compute conflict_trend_pct, with explicit zero-previous-month rule ----
fact_conflict = fact_conflict.sort_values(["location_id", "time_id"])
fact_conflict["prev_month_count"] = fact_conflict.groupby("location_id")["conflict_event_count"].shift(1)

def compute_trend(row):
    if pd.isna(row["prev_month_count"]) or row["prev_month_count"] == 0:
        return pd.NA
    return round(((row["conflict_event_count"] - row["prev_month_count"]) / row["prev_month_count"]) * 100, 2)

fact_conflict["conflict_trend_pct"] = fact_conflict.apply(compute_trend, axis=1)

# ---- 8. Final column selection ----
fact_conflict_final = fact_conflict[[
    "location_id", "time_id", "conflict_event_count", "fatalities", "conflict_density", "conflict_trend_pct"
]].reset_index(drop=True)

# ---- 9. Display and save ----
pd.set_option("display.max_rows", None)
print(fact_conflict_final)
fact_conflict_final.to_csv("fact_conflict_somalia.csv", index=False)

from google.colab import files
files.download("fact_conflict_somalia.csv")

          location_id time_id  conflict_event_count  fatalities  \
0            SO_AADAN  202401                     0           0   
1            SO_AADAN  202402                     3           6   
2          SO_AFGOOYE  202401                    17           7   
3          SO_AFGOOYE  202402                    29          21   
4          SO_AFMADOW  202401                     2           0   
5          SO_AFMADOW  202402                     5           2   
6      SO_BAAR_DHEERE  202401                     0           0   
7      SO_BAAR_DHEERE  202402                     1           1   
8        SO_BADHAADHE  202401                     5           7   
9        SO_BADHAADHE  202402                     1           0   
10          SO_BADHAN  202401                     0           0   
11          SO_BADHAN  202402                     0           0   
12            SO_BAKI  202401                     0           0   
13            SO_BAKI  202402                     0           

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# fact_climate

In [ ]:
import os
import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from rasterstats import zonal_stats

# ---- 0. Sanity check: confirm what's actually in the working directory ----
print("Working directory:", os.getcwd())
print("CHIRPS files found:", [f for f in os.listdir() if f.startswith("chirps")])
print("VHI files found:", [f for f in os.listdir() if f.startswith("VHP")])

# ---- 1. Load Admin2 boundaries -- ALL 74 districts, not just the 35 with market coverage ----
gdf_full = gpd.read_file("gadm41_SOM.gpkg", layer="ADM_ADM_2")
dim_location = pd.read_csv("dim_location_somalia_full74.csv")

gdf = gdf_full.merge(
    dim_location[["location_id", "gadm_gid_2"]],
    left_on="GID_2", right_on="gadm_gid_2"
)[["location_id", "geometry"]]

print(f"Districts in gdf: {len(gdf)}  (expected 74)")

# ---- 2. Confirmed NoData values (from direct pixel inspection, not file metadata) ----
CHIRPS_NODATA = -9999
VHI_NODATA = -9999

# ---- 3. Zonal mean helper ----
def zonal_mean(gdf, tif_path, nodata):
    stats = zonal_stats(gdf, tif_path, stats="mean", nodata=nodata, geojson_out=False)
    return np.array([s["mean"] if s["mean"] is not None else np.nan for s in stats])

# ---- 4. CHIRPS monthly rainfall: direct zonal mean, no aggregation needed ----
rainfall_jan = zonal_mean(gdf, "chirps-v2.0.2024.01.tif", CHIRPS_NODATA)
rainfall_feb = zonal_mean(gdf, "chirps-v2.0.2024.02.tif", CHIRPS_NODATA)

# ---- 5. VHI weekly -> monthly, day-weighted for boundary weeks ----
vhi_files = {i: f"VHP.G04.C07.j01.P2024{i:03d}.VH.VHI.tif" for i in range(1, 10)}
vhi_weekly = {i: zonal_mean(gdf, path, VHI_NODATA) for i, path in vhi_files.items()}

jan_weights = {1: 1, 2: 1, 3: 1, 4: 1, 5: 3/7}
vhi_jan = sum(vhi_weekly[w] * wt for w, wt in jan_weights.items()) / sum(jan_weights.values())

feb_weights = {5: 4/7, 6: 1, 7: 1, 8: 1, 9: 4/7}
vhi_feb = sum(vhi_weekly[w] * wt for w, wt in feb_weights.items()) / sum(feb_weights.values())

# ---- 6. Assemble fact_climate ----
fact_climate = pd.concat([
    pd.DataFrame({
        "location_id": gdf["location_id"], "time_id": "202401",
        "rainfall_mean_mm": rainfall_jan.round(2),
        "vegetation_health_index_mean": vhi_jan.round(2)
    }),
    pd.DataFrame({
        "location_id": gdf["location_id"], "time_id": "202402",
        "rainfall_mean_mm": rainfall_feb.round(2),
        "vegetation_health_index_mean": vhi_feb.round(2)
    }),
], ignore_index=True).sort_values(["location_id", "time_id"]).reset_index(drop=True)

# ---- 7. Display and validate ----
pd.set_option("display.max_rows", None)
print(f"Shape: {fact_climate.shape}  (expected 74 x 2 = 148)")
print(fact_climate.isna().sum())
print(fact_climate)

# ---- 8. Save / download ----
fact_climate.to_csv("fact_climate_somalia.csv", index=False)
from google.colab import files
files.download("fact_climate_somalia.csv")

Working directory: /content
CHIRPS files found: ['chirps-v2.0.2024.02.tif', 'chirps-v2.0.2024.01.tif']
VHI files found: ['VHP.G04.C07.j01.P2024005.VH.VHI.tif', 'VHP.G04.C07.j01.P2024009.VH.VHI.tif', 'VHP.G04.C07.j01.P2024004.VH.VHI.tif', 'VHP.G04.C07.j01.P2024006.VH.VHI.tif', 'VHP.G04.C07.j01.P2024002.VH.VHI.tif', 'VHP.G04.C07.j01.P2024007.VH.VHI.tif', 'VHP.G04.C07.j01.P2024003.VH.VHI.tif', 'VHP.G04.C07.j01.P2024008.VH.VHI.tif', 'VHP.G04.C07.j01.P2024001.VH.VHI.tif']
Districts in gdf: 74  (expected 74)
Shape: (148, 4)  (expected 74 x 2 = 148)
location_id                     0
time_id                         0
rainfall_mean_mm                0
vegetation_health_index_mean    0
dtype: int64
          location_id time_id  rainfall_mean_mm  vegetation_health_index_mean
0            SO_AADAN  202401              2.44                         45.98
1            SO_AADAN  202402              1.38                         45.47
2          SO_AFGOOYE  202401              1.26                     

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os
print(os.path.getsize("chirps-v2.0.2024.01.tif"))

57612550


# fact_reporting

In [ ]:
import pandas as pd
import re

# ---- 1. Load inputs ----
rw = pd.read_csv("reliefweb_somalia_jan_feb_2024.csv")
dim_location = pd.read_csv("dim_location_somalia_full74.csv")
rw["time_id"] = rw["month"].str.replace("-", "")
rw["themes"] = rw["themes"].fillna("")  # untagged reports still count toward the total, not toward theme counts

alias_map = {
    "Banadir": ["Mogadishu"], "Baydhaba": ["Baidoa"],
    "Belet Weyne": ["Beledweyne", "Beletweyne"], "Belet Xaawo": ["Beledhawa", "Belethawa"],
    "Bossaso": ["Bosaso"], "Burco": ["Burao"], "Cabudwaaq": ["Abudwak", "Abudwaq"],
    "Cadaado": ["Adado"], "Ceel Barde": ["El Barde"], "Ceel Waaq": ["El Wak"],
    "Ceerigaabo": ["Erigavo"], "Gaalkacyo": ["Galkayo", "Galcaio"], "Garoowe": ["Garowe"],
    "Hargeysa": ["Hargeisa"], "Kismaayo": ["Kismayo"], "Laas Caanood": ["Las Anod", "Laascaanood"],
}

# ---- 2. For each district and month: match reports, then split by theme ----
months = ["202401", "202402"]
rows = []
for _, loc in dim_location.iterrows():
    admin2 = loc["admin2"]
    names_to_check = [admin2] + alias_map.get(admin2, [])
    pattern = "|".join(re.escape(n) for n in names_to_check)

    for time_id in months:
        month_reports = rw[rw["time_id"] == time_id].copy()
        mentions_district = (
            month_reports["title"].str.contains(pattern, case=False, na=False, regex=True) |
            month_reports["body"].str.contains(pattern, case=False, na=False, regex=True)
        )
        matched = month_reports[mentions_district]

        rows.append({
            "location_id": loc["location_id"],
            "time_id": time_id,
            "reports_mentioning_district_count": len(matched),
            "food_nutrition_report_count": matched["themes"].str.contains("Food and Nutrition").sum(),
            "wash_report_count": matched["themes"].str.contains("Water Sanitation Hygiene").sum(),
        })

fact_reporting = pd.DataFrame(rows)

# ---- 3. Display and save ----
pd.set_option("display.max_rows", None)
print(fact_reporting)
fact_reporting.to_csv("fact_reporting_somalia.csv", index=False)

from google.colab import files
files.download("fact_reporting_somalia.csv")

          location_id time_id  reports_mentioning_district_count  \
0            SO_AADAN  202401                                  0   
1            SO_AADAN  202402                                  0   
2          SO_AFGOOYE  202401                                  5   
3          SO_AFGOOYE  202402                                  1   
4          SO_AFMADOW  202401                                  9   
5          SO_AFMADOW  202402                                  3   
6      SO_BAAR_DHEERE  202401                                  0   
7      SO_BAAR_DHEERE  202402                                  0   
8        SO_BADHAADHE  202401                                  2   
9        SO_BADHAADHE  202402                                  1   
10          SO_BADHAN  202401                                  1   
11          SO_BADHAN  202402                                  4   
12            SO_BAKI  202401                                  0   
13            SO_BAKI  202402                   

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# fact_market

In [ ]:
import pandas as pd

# ---- 1. Load and filter WFP price data ----
df = pd.read_csv("Prices-Export-Tue Jul 28 2026 21_38_31 GMT+0100 (British Summer Time).csv")
dim_location = pd.read_csv("dim_location_somalia_full74.csv")

som = df[(df["Country"] == "Somalia") & (df["Data Type"] == "Aggregated")].copy()
som["Price Date"] = pd.to_datetime(som["Price Date"], format="%d/%m/%Y")
som["time_id"] = som["Price Date"].dt.strftime("%Y%m")

# ---- 2. Exchange rate lookup: Admin2 x Month -> rate ----
fx = som[som["Commodity"] == "Exchange rate"][["Admin 2", "time_id", "Price"]].rename(
    columns={"Admin 2": "admin2", "Price": "exchange_rate_to_usd"})

# ---- 3. Filter to the 4 basket commodities, map to short names ----
commodity_short = {
    "Wheat flour (imported)": "wheatflour",
    "Rice (imported)": "rice",
    "Sugar (white)": "sugar",
    "Oil (vegetable, imported)": "oil",
}
prices = som[som["Commodity"].isin(commodity_short.keys())].copy()
prices["commodity"] = prices["Commodity"].map(commodity_short)
prices = prices.rename(columns={"Admin 2": "admin2", "Price": "price_local", "Pewi": "pewi_score"})

# ---- 4. Convert to USD using same-district, same-month exchange rate ----
prices = prices.merge(fx, on=["admin2", "time_id"], how="left")
prices["price_usd"] = (prices["price_local"] / prices["exchange_rate_to_usd"]).round(4)

# ---- 5. Build complete Admin2 x Month spine (74 districts x 2 months), pivot commodities directly into it ----
months = pd.DataFrame({"time_id": ["202401", "202402"]})
spine = dim_location[["location_id", "admin2"]].merge(months, how="cross")

price_pivot = prices.pivot_table(index=["admin2", "time_id"], columns="commodity", values="price_usd", aggfunc="mean")
price_pivot.columns = [f"{c}_price_usd" for c in price_pivot.columns]

pewi_pivot = prices.pivot_table(index=["admin2", "time_id"], columns="commodity", values="pewi_score", aggfunc="mean")
pewi_pivot.columns = [f"{c}_pewi_score" for c in pewi_pivot.columns]

fact_market = spine.merge(price_pivot.reset_index(), on=["admin2", "time_id"], how="left")
fact_market = fact_market.merge(pewi_pivot.reset_index(), on=["admin2", "time_id"], how="left")

# ---- 6. price_change_pct, per commodity, computed directly on the wide table ----
fact_market = fact_market.sort_values(["location_id", "time_id"])
for commodity in commodity_short.values():
    price_col = f"{commodity}_price_usd"
    fact_market[f"{commodity}_price_change_pct"] = (
        fact_market.groupby("location_id")[price_col].pct_change().mul(100).round(2)
    )

# ---- 7. Final column order ----
ordered_cols = ["location_id", "time_id"]
for commodity in commodity_short.values():
    ordered_cols += [f"{commodity}_price_usd", f"{commodity}_price_change_pct", f"{commodity}_pewi_score"]
fact_market_final = fact_market[ordered_cols].reset_index(drop=True)

# ---- 8. Display and save ----
pd.set_option("display.max_columns", None)
print(f"Shape: {fact_market_final.shape}  (expected 74 x 2 = 148)")
print(fact_market_final)
fact_market_final.to_csv("fact_market_somalia.csv", index=False)

from google.colab import files
files.download("fact_market_somalia.csv")

Shape: (148, 14)  (expected 74 x 2 = 148)
          location_id time_id  wheatflour_price_usd  \
0            SO_AADAN  202401                   NaN   
1            SO_AADAN  202402                   NaN   
2          SO_AFGOOYE  202401                   NaN   
3          SO_AFGOOYE  202402                   NaN   
4          SO_AFMADOW  202401                0.7368   
5          SO_AFMADOW  202402                0.6600   
6      SO_BAAR_DHEERE  202401                   NaN   
7      SO_BAAR_DHEERE  202402                   NaN   
8        SO_BADHAADHE  202401                   NaN   
9        SO_BADHAADHE  202402                   NaN   
10          SO_BADHAN  202401                   NaN   
11          SO_BADHAN  202402                   NaN   
12            SO_BAKI  202401                   NaN   
13            SO_BAKI  202402                   NaN   
14          SO_BALCAD  202401                   NaN   
15          SO_BALCAD  202402                   NaN   
16         SO_BANADIR  

/tmp/ipykernel_3282/2633393279.py:48: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fact_market.groupby("location_id")[price_col].pct_change().mul(100).round(2)
/tmp/ipykernel_3282/2633393279.py:48: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fact_market.groupby("location_id")[price_col].pct_change().mul(100).round(2)
/tmp/ipykernel_3282/2633393279.py:48: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=N

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Merging into one final dataset

In [ ]:
import pandas as pd

# ---- 1. Load all tables ----
dim_location = pd.read_csv("dim_location_somalia_full74.csv")
fact_conflict = pd.read_csv("fact_conflict_somalia.csv", dtype={"time_id": str})
fact_climate = pd.read_csv("fact_climate_somalia.csv", dtype={"time_id": str})
fact_market = pd.read_csv("fact_market_somalia.csv", dtype={"time_id": str})
fact_reporting = pd.read_csv("fact_reporting_somalia.csv", dtype={"time_id": str})

# ---- 2. Build the spine: Admin2 x Month (74 x 2 = 148 rows) ----
months = pd.DataFrame({"time_id": ["202401", "202402"]})
spine = dim_location[["location_id", "country_code", "admin1", "admin2", "has_market_coverage"]].merge(
    months, how="cross"
)

# ---- 3. Merge every mechanism directly -- all four are now at the same grain ----
panel = spine.merge(fact_conflict, on=["location_id", "time_id"], how="left")
panel = panel.merge(fact_climate, on=["location_id", "time_id"], how="left")
panel = panel.merge(fact_market, on=["location_id", "time_id"], how="left")
panel = panel.merge(fact_reporting, on=["location_id", "time_id"], how="left")

# ---- 4. Validate ----
print(f"Shape: {panel.shape}  (expected 148 rows)")
print(panel.isna().sum())
print(panel)

# ---- 5. Save and download ----
panel.to_csv("vw_food_insecurity_panel.csv", index=False)

from google.colab import files
files.download("vw_food_insecurity_panel.csv")

Shape: (148, 27)  (expected 148 rows)
location_id                            0
country_code                           0
admin1                                 0
admin2                                 0
has_market_coverage                    0
time_id                                0
conflict_event_count                   0
fatalities                             0
conflict_density                       0
conflict_trend_pct                    99
rainfall_mean_mm                       0
vegetation_health_index_mean           0
wheatflour_price_usd                  79
wheatflour_price_change_pct          114
wheatflour_pewi_score                 85
rice_price_usd                        79
rice_price_change_pct                114
rice_pewi_score                       86
sugar_price_usd                       79
sugar_price_change_pct               114
sugar_pewi_score                      85
oil_price_usd                         79
oil_price_change_pct                 114
oil_pewi_score     

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Performing experiment

Two experiments are presented below to demonstrate that the integrated dataset supports genuine cross-mechanism analysis

**A disclaimer before proceeding**: these experiments are included for testing and illustrative purposes only. They are not intended to support any substantive conclusion about food insecurity, conflict, or market dynamics in Somalia. The prototype covers a single country and only two months of data, which is far too limited a sample to draw reliable inferences from. The purpose here is solely to confirm that the integrated dataset behaves as expected and can support the type of cross-mechanism analysis it was designed for.]

##Experiment 1: Market Coverage and Conflict

###  Background

While building the dataset, we found that only 35 of Somalia's 74 districts have a WFP-monitored market. The other 39 have no price data at all — not due to a collection error, but because no market exists there for WFP to monitor..

### Question

This raises a natural concern: are districts without a market simply random, or do they tend to be the more conflict-affected ones? If the latter were true, it would mean the market data is missing in a biased way, quietly excluding the places most likely to be struggling. The question tested here is: **do districts with a market look different, in terms of conflict, from districts without one?**

In [ ]:
import pandas as pd
from scipy import stats

panel = pd.read_csv("vw_food_insecurity_panel.csv", dtype={"time_id": str})

covered = panel[panel["has_market_coverage"] == True]
not_covered = panel[panel["has_market_coverage"] == False]

print(f"Districts with a market: {covered['location_id'].nunique()}")
print(f"Districts without a market: {not_covered['location_id'].nunique()}")
print()

for col in ["conflict_density", "fatalities", "conflict_event_count"]:
    stat, p = stats.mannwhitneyu(covered[col].dropna(), not_covered[col].dropna(), alternative="two-sided")
    print(f"{col}:")
    print(f"  avg with market:    {covered[col].mean():.4f}")
    print(f"  avg without market: {not_covered[col].mean():.4f}")
    print(f"  p-value: {p:.4f}")
    print()

Districts with a market: 35
Districts without a market: 39

conflict_density:
  avg with market:    0.0067
  avg without market: 0.0012
  p-value: 0.3554

fatalities:
  avg with market:    4.6143
  avg without market: 7.3718
  p-value: 0.4733

conflict_event_count:
  avg with market:    4.5714
  avg without market: 3.2436
  p-value: 0.1786



### Method

The 74 districts were split into two groups based on market coverage, and three conflict measures were compared between them: event count, fatalities, and conflict density.

Comparing the two averages directly isn't enough on its own — any two groups will show some difference just by chance, so that alone doesn't tell you much. The real question is whether the difference is bigger than what ordinary randomness would produce. That's what the statistical test checks. We used the Mann-Whitney U test specifically, since conflict data tends to be skewed (most districts have very few events, a handful have many), which this test handles better than one that assumes a symmetric distribution. It returns a p-value: small means the difference is likely real, large means it's within the range you'd expect from noise alone.

### Results

All three p-values came back above the standard 0.05 threshold (0.36, 0.47, and 0.18), meaning none of the differences look bigger than ordinary chance would produce. The direction wasn't even consistent: market districts had a slightly higher average event count, but a lower average fatality count than non-market districts — the kind of scattered pattern you'd expect from noise, not a real effect.

### Conclusion

No evidence was found that WFP's market coverage is systematically biased toward less conflict-affected districts. This is a reassuring result, but a modest one — with only 74 districts and two months of data, the test has limited power and would likely miss anything short of a fairly large effect. The fair conclusion is narrow: this specific bias was tested for and not found, not that no such bias exists at all.

## Experiment 2: Does Reporting Attention Track Conflict?

### Background

A feature was constructed earlier that counts how many humanitarian reports mention each district by name (`reports_mentioning_district_count`), derived by searching report text for district names. Since this feature and the conflict data now sit in the same table, it is possible to test whether the two are related.

### Question

Do districts with more recorded conflict also receive more humanitarian reporting attention? A positive relationship would also serve as a useful check on the reporting feature itself, indicating that the district-matching approach captures a genuine signal rather than noise.

In [ ]:
from scipy import stats
import pandas as pd

panel = pd.read_csv("vw_food_insecurity_panel.csv", dtype={"time_id": str})

r, p = stats.spearmanr(panel['conflict_event_count'], panel['reports_mentioning_district_count'])
print(f"Conflict vs reports (same month): r={r:.3f}  p={p:.4f}  n={len(panel)}")

Conflict vs reports (same month): r=0.318  p=0.0001  n=148


### Initial result

A statistically significant positive correlation was found (r = 0.32, p < 0.001): districts with more conflict tend to receive more reporting attention. Before treating this as reliable, it was checked against two possible sources of distortion.

### Check 1: Is this driven by a single district?

Banadir (Mogadishu) is the capital, and would be expected to show both high conflict and high reporting volume simply due to its size and prominence. If Banadir alone were driving the result, the finding would say more about one city than about a general pattern.

In [ ]:
no_banadir = panel[panel['location_id'] != 'SO_BANADIR']
r2, p2 = stats.spearmanr(no_banadir['conflict_event_count'], no_banadir['reports_mentioning_district_count'])
print(f"Without Banadir: r={r2:.3f}  p={p2:.4f}  n={len(no_banadir)}")

Without Banadir: r=0.286  p=0.0005  n=146


Excluding Banadir changed the correlation only slightly (0.32 to 0.29, still p < 0.001), indicating the relationship holds broadly across districts rather than being an artefact of one outlier.

### Check 2: Is the relationship stable across months?

With only two months of data available, it is worth checking whether the relationship holds consistently, or whether it is driven disproportionately by a single month.

In [ ]:
for t in panel['time_id'].unique():
    sub = panel[panel['time_id'] == t]
    r3, p3 = stats.spearmanr(sub['conflict_event_count'], sub['reports_mentioning_district_count'])
    print(f"Month {t}: r={r3:.3f}  p={p3:.4f}  n={len(sub)}")

Month 202401: r=0.475  p=0.0000  n=74
Month 202402: r=0.137  p=0.2443  n=74


The relationship was strong in January (r = 0.48, p < 0.001) but not statistically significant in February alone (r = 0.14, p = 0.24). This raised the question of whether the overall result was genuinely reliable, or an artefact of one strong month.

### A possible explanation: reporting lag

Humanitarian reports are unlikely to be published the instant an event occurs; there is typically a delay between a conflict event and any resulting coverage. This suggests that a given month's conflict may predict the *following* month's reporting better than reporting in the same month. This was tested directly.

In [ ]:
jan = panel[panel['time_id'] == panel['time_id'].unique()[0]][['location_id', 'conflict_event_count']].rename(
    columns={'conflict_event_count': 'jan_conflict'})
feb = panel[panel['time_id'] == panel['time_id'].unique()[1]][['location_id', 'reports_mentioning_district_count']].rename(
    columns={'reports_mentioning_district_count': 'feb_reports'})
lagged = jan.merge(feb, on='location_id')

r4, p4 = stats.spearmanr(lagged['jan_conflict'], lagged['feb_reports'])
print(f"January conflict -> February reports: r={r4:.3f}  p={p4:.4f}  n={len(lagged)}")

January conflict -> February reports: r=0.384  p=0.0007  n=74


### Result

January's conflict level predicted February's reporting volume more strongly than same-month conflict predicted same-month reporting (r = 0.38–0.39, p < 0.001). This supports the reporting-lag explanation and accounts for why the same-month relationship appeared inconsistent across the two available months.

### Conclusion

Three findings together support a genuine relationship rather than a coincidental one: the correlation between conflict and reporting attention is statistically significant, it is not driven by a single district, and its apparent instability across months is explained by a plausible reporting lag rather than by the relationship being spurious. This result also demonstrates a practical value of integration: the pattern could only be identified by combining conflict data and reporting data within the same table, as neither source alone would reveal it.

# Week of August 10, 2026

---

# Task 1 & 2: Reframing reaserch question and defining sub questions

## Research Question

**Main RQ**: How unevenly do humanitarian data sources observe different places and crisis mechanisms, and what are the consequences for monitoring humanitarian conditions more broadly?

**Sub-questions**:

1. **Where are the gaps, and are they systematic?** Which mechanisms (conflict, climate, market, reporting) have the largest spatial coverage gaps, and are these gaps randomly distributed or associated with district characteristics such as urbanization, accessibility, or conflict exposure? Does combining multiple sources meaningfully reduce the gaps present in any single source?

2. **Does observability differ by crisis type, not just by place?** Does humanitarian reporting respond proportionally across different kinds of crisis? , for instance, is conflict-driven distress more consistently captured than slow-onset climate stress of comparable severity?

   For example, let's suppose two districts are going through severe conditions. One is dealing with extreme violence, while the other is in the middle of a serious drought.Does the violent one get  more attention just because it's more dramatic, while the drought quietly gets ignored even though it might be just as damaging to people?

3. **Does poor observability translate into being wrong, not just incomplete?** In districts where the dataset shows weaker coverage, does the dataset's derived assessment diverge more sharply from IPC's official food-insecurity classification than it does in well-observed districts?


## Sub-Question → Analysis → Status

| Sub-question | Corresponding analysis | Status |
|---|---|---|
| **1. Where are the gaps, and are they systematic?** | (a) Compare coverage % across mechanisms — done via Data Quality Evaluation. (b) Test whether market coverage correlates with conflict exposure — done (Pilot Analysis 1, no significant relationship found). (c) Test correlation with urbanization/accessibility — **not started**, requires new features not currently in the dataset. (d) Test whether combining sources reduces gaps — **not started**, but testable now with existing data. | **Partially covered** |
| **2. Does observability differ by crisis type?** | Compare reporting attention during conflict-crisis district-months vs. climate-crisis district-months of comparable severity. | **Not testable with current window** — VHI shows no meaningful drought signal in Jan–Feb 2024 (130 of 148 district-months show no vegetation stress at all). Requires a window that includes a genuine climate-stress period. |
| **3. Does poor observability translate into being wrong?** | Compare the dataset's derived assessment against IPC classifications, split by observability level. | **Not started** — deliberately deferred; IPC data not yet sourced (see earlier IPC availability check). |

# Task 3: Organising commodity specific features as one feature family and evaluating their missingness and redundancy.

In [ ]:
import pandas as pd

panel = pd.read_csv("vw_food_insecurity_panel.csv", dtype={"time_id": str})

commodities = ["wheatflour", "rice", "sugar", "oil"]
price_cols = [f"{c}_price_usd" for c in commodities]
change_cols = [f"{c}_price_change_pct" for c in commodities]
pewi_cols = [f"{c}_pewi_score" for c in commodities]

# ---- 1. Missingness across the market feature family ----
print("=== Missingness, all 12 market features (out of 148) ===")
for group_name, cols in [("Price (USD)", price_cols), ("Price change %", change_cols), ("Pewi score", pewi_cols)]:
    print(f"\n{group_name}:")
    for c in cols:
        n_missing = panel[c].isna().sum()
        print(f"  {c:35s} {n_missing:3d}/148 missing ({n_missing/148*100:.1f}%)")

# ---- 2. Redundancy: correlation between commodities' price levels ----
print("\n=== Correlation between commodities' PRICE levels ===")
print(panel[price_cols].corr(min_periods=10).round(2))

# ---- 3. Redundancy: correlation between commodities' Pewi scores ----
print("\n=== Correlation between commodities' PEWI scores ===")
print(panel[pewi_cols].corr(min_periods=10).round(2))

# ---- 4. Sample size behind each correlation (how many districts had both commodities' data) ----
print("\n=== Pairwise N (non-null overlap) for price correlations ===")
for i, c1 in enumerate(price_cols):
    for c2 in price_cols[i+1:]:
        n = panel[[c1, c2]].dropna().shape[0]
        print(f"  {c1} vs {c2}: n={n}")

=== Missingness, all 12 market features (out of 148) ===

Price (USD):
  wheatflour_price_usd                 79/148 missing (53.4%)
  rice_price_usd                       79/148 missing (53.4%)
  sugar_price_usd                      79/148 missing (53.4%)
  oil_price_usd                        79/148 missing (53.4%)

Price change %:
  wheatflour_price_change_pct         114/148 missing (77.0%)
  rice_price_change_pct               114/148 missing (77.0%)
  sugar_price_change_pct              114/148 missing (77.0%)
  oil_price_change_pct                114/148 missing (77.0%)

Pewi score:
  wheatflour_pewi_score                85/148 missing (57.4%)
  rice_pewi_score                      86/148 missing (58.1%)
  sugar_pewi_score                     85/148 missing (57.4%)
  oil_pewi_score                       85/148 missing (57.4%)

=== Correlation between commodities' PRICE levels ===
                      wheatflour_price_usd  rice_price_usd  sugar_price_usd  \
wheatflour_price_usd 

## Market Feature Family: Missingness and Redundancy

Following supervisor feedback, the 12 commodity-level market features are organized here as a single coherent family (4 commodities × price, price change, Pewi score) and evaluated jointly, rather than treated as 12 independent decisions.

**Missingness**: All four commodities' price features share identical missingness (79 of 148 district-months, 53.4%), since all four depend on the same constraint — whether a market exists in that district at all — rather than on anything commodity-specific. Pewi scores diverge marginally (rice: 86 missing vs. 85 for the others), reflecting WFP's own seasonal-baseline coverage.

**Redundancy**: Correlating each commodity's price and Pewi score against the others (n=69, the districts with market coverage) reveals a clear asymmetry. Wheat flour and rice are highly correlated (r=0.89 for price, r=0.61 for Pewi) — both are imported staples plausibly driven by shared exchange-rate and import-cost dynamics, making this pair the strongest candidate for consolidation if the feature set needs to shrink further. Sugar, by contrast, is nearly uncorrelated with every other commodity (r=0.02–0.21 across both price and Pewi), confirming statistically what the earlier sugar price anomaly investigation suggested qualitatively: sugar carries genuinely independent information that would be lost if commodities were combined into a single index.

**Conclusion**: all four commodities are retained for the prototype, consistent with supervisor guidance. Wheat flour and rice are flagged as the most likely candidates for consolidation into a single "imported cereal" measure in the harmonized cross-country layer (see next section), while sugar's independence is treated as a specific justification for keeping at least one commodity fully standalone rather than only ever working with an aggregate.

# Task 4: Design a harmonised market-anomaly layer for future cross-country comparison.


In [ ]:
import pandas as pd
import numpy as np

panel = pd.read_csv("vw_food_insecurity_panel.csv", dtype={"time_id": str})

commodities = ["wheatflour", "rice", "sugar", "oil"]
pewi_cols = [f"{c}_pewi_score" for c in commodities]

# ---- 1. commodities_tracked_count: how many commodities have data this district-month ----
panel["commodities_tracked_count"] = panel[pewi_cols].notna().sum(axis=1)

# ---- 2. market_anomaly_mean: average Pewi across available commodities ----
panel["market_anomaly_mean"] = panel[pewi_cols].mean(axis=1, skipna=True).round(3)

# ---- 3. market_anomaly_max: highest Pewi among available commodities ----
has_any = panel[pewi_cols].notna().any(axis=1)
panel["market_anomaly_max"] = np.nan
panel.loc[has_any, "market_anomaly_max"] = panel.loc[has_any, pewi_cols].max(axis=1).round(3)

# ---- 4. market_stress_count: commodities above an approximate stress threshold ----
# NOTE: WFP's exact ALPS thresholds were not available; 1.0 is a documented modeling choice
STRESS_THRESHOLD = 1.0
panel["market_stress_count"] = (panel[pewi_cols] > STRESS_THRESHOLD).sum(axis=1)

# ---- Validation: missingness ----
print("=== Missingness of the new features ===")
for c in ["commodities_tracked_count", "market_anomaly_mean", "market_anomaly_max", "market_stress_count"]:
    print(f"{c:32s} missing: {panel[c].isna().sum():3d} / 148")

print("\n=== commodities_tracked_count distribution ===")
print(panel["commodities_tracked_count"].value_counts().sort_index())

print("\n=== market_stress_count distribution ===")
print(panel["market_stress_count"].value_counts().sort_index())

panel.to_csv("vw_food_insecurity_panel.csv", index=False)

=== Missingness of the new features ===
commodities_tracked_count        missing:   0 / 148
market_anomaly_mean              missing:  85 / 148
market_anomaly_max               missing:  85 / 148
market_stress_count              missing:   0 / 148

=== commodities_tracked_count distribution ===
commodities_tracked_count
0    85
3     1
4    62
Name: count, dtype: int64

=== market_stress_count distribution ===
market_stress_count
0    116
1     23
2      8
3      1
Name: count, dtype: int64


## Harmonized Market-Anomaly Layer

Four features were added on top of the existing commodity-specific columns, designed to be computable regardless of which specific commodities a country tracks -- intended for cross-country comparison once Ethiopia, Kenya, and South Sudan are integrated.

- `commodities_tracked_count` -- how many of the tracked commodities have data for that district-month
- `market_anomaly_mean` -- average Pewi score across available commodities
- `market_anomaly_max` -- the single highest Pewi score among available commodities
- `market_stress_count` -- count of commodities with Pewi score above 1.0 (an approximate threshold, since WFP's exact ALPS cutoffs were not available; documented as a modeling choice)

A fifth candidate feature, identifying which commodity produced the maximum anomaly score, was considered and dropped: it does not serve either sub-question directly (see mapping below) and is purely explanatory rather than analytical. If needed later, this information remains recoverable by inspecting the underlying commodity-specific columns directly.

**Missingness**: `commodities_tracked_count` and `market_stress_count` are never missing, since a lack of market data legitimately produces a value of zero rather than an unknown. `market_anomaly_mean` and `market_anomaly_max` inherit the same missingness as the underlying Pewi scores (85 of 148 district-months).

**Coverage pattern**: commodity reporting is close to all-or-nothing in this data -- 62 district-months report all 4 commodities, 85 report none, and only 1 reports a partial set of 3.

### Which sub-question each feature actually serves

| Feature | Serves | Why |
|---|---|---|
| `commodities_tracked_count` | **Sub-question 1** (observability gaps) | Refines `has_market_coverage` from a binary flag into a graded measure -- a market reporting all 4 commodities is more fully observed than one reporting only 1 or 2. Directly strengthens the "coverage is not the same as true observability" distinction (Limitation #7). |
| `market_anomaly_mean` | **Sub-question 3** (does poor observability lead to being wrong) | Provides a summary "derived condition" from the dataset that can be compared against IPC's classification once that data is sourced. |
| `market_anomaly_max` | **Sub-question 3** | Captures a single-commodity spike that an average might dilute, giving a second, complementary basis for comparison against IPC. |
| `market_stress_count` | **Sub-question 3** | A third, count-based way of summarizing conditions for the same IPC comparison -- useful to have multiple candidate summary measures rather than relying on just one. |

# Task 5: Rename and critically test area-based conflict density; remove it if it adds little beyond event counts.

In [ ]:
import pandas as pd
from scipy import stats

panel = pd.read_csv("vw_food_insecurity_panel.csv", dtype={"time_id": str})
dim_location = pd.read_csv("dim_location_somalia_full74.csv")
merged = panel.merge(dim_location[["location_id", "area_km2"]], on="location_id", how="left")

print("=== Full sample (148 district-months) ===")
r1, p1 = stats.spearmanr(merged["area_km2"], merged["conflict_density"])
print(f"area_km2 vs conflict_density: r={r1:.3f}  p={p1:.4f}")
r2, p2 = stats.spearmanr(merged["conflict_event_count"], merged["conflict_density"])
print(f"conflict_event_count vs conflict_density: r={r2:.3f}  p={p2:.4f}")

print("\n=== Restricted to districts with at least 1 event (92 rows) ===")
nonzero = merged[merged["conflict_event_count"] > 0]
r3, p3 = stats.spearmanr(nonzero["conflict_event_count"], nonzero["conflict_density"])
print(f"conflict_event_count vs conflict_density: r={r3:.3f}  p={p3:.4f}")
r4, p4 = stats.spearmanr(nonzero["area_km2"], nonzero["conflict_density"])
print(f"area_km2 vs conflict_density: r={r4:.3f}  p={p4:.4f}")

print("\n=== Does density add anything beyond count for predicting severity? ===")
sub = merged.dropna(subset=["fatalities"])
r5, _ = stats.spearmanr(sub["conflict_event_count"], sub["fatalities"])
r6, _ = stats.spearmanr(sub["conflict_density"], sub["fatalities"])
print(f"conflict_event_count vs fatalities: r={r5:.3f}")
print(f"conflict_density vs fatalities:     r={r6:.3f}")

# ---- Drop the feature from the panel ----
panel = panel.drop(columns=["conflict_density"])
panel.to_csv("vw_food_insecurity_panel.csv", index=False)
print("\nconflict_density removed. Feature count: 20.")

=== Full sample (148 district-months) ===
area_km2 vs conflict_density: r=-0.339  p=0.0000
conflict_event_count vs conflict_density: r=0.962  p=0.0000

=== Restricted to districts with at least 1 event (92 rows) ===
conflict_event_count vs conflict_density: r=0.850  p=0.0000
area_km2 vs conflict_density: r=-0.644  p=0.0000

=== Does density add anything beyond count for predicting severity? ===
conflict_event_count vs fatalities: r=0.776
conflict_density vs fatalities:     r=0.727

conflict_density removed. Feature count: 20.


## Reviewing conflict_density

`conflict_density` was renamed to `conflict_events_per_1000_km2` to more accurately describe what it measures, and tested to check whether it adds anything beyond the raw event count it is derived from.

**What we found**: `conflict_events_per_1000_km2` correlates very closely with `conflict_event_count` -- at 0.96 across all 148 district-months, and still at 0.85 when limited to only the 92 district-months that had at least one recorded event. This second check matters, since it rules out the high correlation simply being an artifact of many districts sharing a value of zero for both measures. The feature also correlates strongly and negatively with district area (-0.34 overall, -0.64 among conflict-affected districts), confirming that it is substantially shaped by the size of the administrative unit rather than capturing something independent. It also does not outperform the raw event count when related to conflict severity: correlation with fatalities is 0.776 for event count versus 0.727 for density.

**Decision**: `conflict_events_per_1000_km2` is removed from the feature set. `conflict_event_count` carries essentially the same information without the area-driven distortion. This reverses an earlier decision to keep the feature, made on the reasoning that it would distinguish concentrated urban conflict from sparse rural conflict -- the data does not support that this distinction actually holds in practice, so the feature is dropped rather than retained on the strength of its original rationale alone.

# Task 6: Review percentage-change features and implement more stable alternatives where necessary.

In [ ]:
import pandas as pd
import numpy as np

panel = pd.read_csv("vw_food_insecurity_panel.csv", dtype={"time_id": str})
panel = panel.sort_values(["location_id", "time_id"])

# ---- Conflict: replace conflict_trend_pct with the log-difference version ----
panel["prev_event_count"] = panel.groupby("location_id")["conflict_event_count"].shift(1)
panel["conflict_trend_pct"] = np.log1p(panel["conflict_event_count"]) - np.log1p(panel["prev_event_count"])
# NOTE: this column is now a log-difference despite the name -- renamed below for clarity
panel = panel.rename(columns={"conflict_trend_pct": "conflict_trend_log"})

# ---- Market: replace each commodity's price_change_pct with its log-difference version ----
commodities = ["wheatflour", "rice", "sugar", "oil"]
for c in commodities:
    price_col = f"{c}_price_usd"
    prev_col = f"prev_{c}_price"
    panel[prev_col] = panel.groupby("location_id")[price_col].shift(1)
    panel[f"{c}_price_change_pct"] = np.log(panel[price_col]) - np.log(panel[prev_col])
    panel = panel.rename(columns={f"{c}_price_change_pct": f"{c}_trend_log"})

# ---- Clean up helper columns ----
panel = panel.drop(columns=["prev_event_count"] + [f"prev_{c}_price" for c in commodities])

print("Feature count unchanged -- same 5 trend columns, now log-difference instead of percentage")
panel.to_csv("vw_food_insecurity_panel.csv", index=False)

Feature count unchanged -- same 5 trend columns, now log-difference instead of percentage


## Reviewing the Trend Features for Stability

`conflict_trend_pct` and the four commodity `_price_change_pct` features were reviewed for a known instability: percentage change is undefined when the previous value is zero, and becomes disproportionately large when the previous value is small (e.g. SO_BARAAWE's conflict count moving from 1 to 7 produced a +600% trend value).

A log-difference alternative was tested and found superior on both counts: missingness dropped from 99/148 to 74/148 for the conflict trend (the remaining 74 are genuinely unavoidable first-month observations), and extreme values were compressed to a reasonable, comparable scale without losing their relative ranking (the same SO_BARAAWE case becomes 1.39; the largest sugar price anomaly, +709%, becomes 2.09 -- still clearly the largest change in the data, just no longer an implausible magnitude).

**Decision**: rather than retaining both versions and increasing the feature count, the percentage-based trend features were replaced by their log-difference equivalents (`conflict_trend_log`, `{commodity}_trend_log`), keeping the total feature count unchanged at 21. The percentage-based calculation is documented here as the version originally used and the reasoning for moving away from it, rather than kept as a live feature alongside its replacement.

# Task 7: Manually validate ReliefWeb district geoparsing and report precision and recall.

In [ ]:
import pandas as pd
import re

rw = pd.read_csv("reliefweb_somalia_jan_feb_2024.csv")
dim_location = pd.read_csv("dim_location_somalia_full74.csv")

# ---- Final merged alias dictionary (WFP crosswalk + ACLED crosswalk + GADM VARNAME_2 + manual research) ----
final_alias_map = {
    "Aadan": ["Adan Yabaal", "Adan Yabal", "Aden Yabal"],
    "Afgooye": ["Afgoi", "Afgoye"],
    "Afmadow": ["Af-madow", "Afamadow", "Afmaadu", "Afmadoow", "Afmadou", "Afmadu"],
    "Baar-Dheere": ["Baardheere", "Bardera", "Bardhere"],
    "Badhaadhe": ["Badhadhe"], "Badhan": ["Las Qoray"], "Balcad": ["Balad", "Ballcad"],
    "Banadir": ["Mogadishu"], "Bander-Beyla": ["Bander Beila", "Bender Bayla"],
    "Baraawe": ["Braawe"], "Baydhaba": ["Baidoa"],
    "Belet Weyne": ["Beledweyne", "Beletweyne"],
    "Belet Xaawo": ["Beledhawa", "Belet Hawa", "Belethawa", "Beletxawa", "Bula-hawa"],
    "Borama": ["Boramo"], "Bossaso": ["Bosaso"], "Bu'aale": ["Buale"],
    "Bulo Burto": ["Bula-Brif", "Bulo-Burte", "Buuloburd"], "Burco": ["Burao"],
    "Buuhoodle": ["Buhodle", "Buuhodle"],
    "Buur Xakaba": ["Bur Hacaba", "Burhakaba", "Buur Hakaba", "Buurhakab"],
    "Cabudwaaq": ["Abudwak", "Abudwaq"], "Cadaado": ["Adado"],
    "Cadale": ["Adale", "Caadale"], "Calawla": ["Alula", "Caluula"],
    "Caynabo": ["Ainabo", "Aynabo", "Caynaba"], "Ceel Barde": ["El Barde"],
    "Ceel Buur": ["Ceelbur", "El Bur"], "Ceel Dheer": ["Ceeldeer", "El Dere"],
    "Ceel Waaq": ["Ceelwaaq", "El Wak", "El Waq"],
    "Ceel-Afwein": ["Ceel Afwayn", "Ceel Afweyn", "Ceelafyeyn", "El Afwe"],
    "Ceerigaabo": ["Ceerigabo", "Erigabo", "Erigavo"],
    "Dhuusamareeb": ["Dhusa-Mareb", "Dhuusamarreeb"], "Diinsoor": ["Dinsor"],
    "Gaalkacyo": ["Galcaio", "Galkacyo", "Galkayo"], "Gabiley": ["Gebiley"],
    "Garbahaaray": ["Garbahaarey", "Garbahaarreey", "Garbaharey"],
    "Garoowe": ["Garowe"], "Goldogob": ["Galdogob"], "Hargeysa": ["Hargeisa"],
    "Hobyo": ["Obbia"], "Iskushuban": ["Iskhushuban"], "Jamaame": ["Jamame"],
    "Jariiban": ["Jarban", "Jeriban"], "Kismaayo": ["Kismayo"],
    "Kuntuwaaray": ["Kurtun Warrey", "Kurtunwaarey"],
    "Laas Caanood": ["Laascaanood", "Las Anod", "Lasanod"], "Lughaya": ["Lughaye"],
    "Luuq": ["Lugh"], "Marka": ["Mark Afgooye"],
    "Oodweyne": ["Odweine", "Odwenyen", "Oodwayne", "Owdweyne"],
    "Qandala": ["Kandala"], "Qansax Dheere": ["Qansadhere"], "Qardho": ["Gardo"],
    "Qoryooley": ["Qoryoley", "Qoryoyley"], "Saakow": ["Sakow"], "Sablale": ["Sablaale"],
    "Sheekh": ["Sheik", "Sheikh"], "Taleex": ["Taleh", "Telex"],
    "Tiyeeglow": ["Tayeeglow", "Tiyeglow"], "Wajid": ["Waajid"],
    "Wanla Weyn": ["Wanlaweyne", "Wanle Weyne"],
    "Xarardheere": ["Haradhere", "Harardheere"], "Xudun": ["Hudun"], "Xudur": ["Hudur"],
    "Zeylac": ["Saylac"],
}

all_names = []
for _, loc in dim_location.iterrows():
    all_names.extend([loc["admin2"]] + final_alias_map.get(loc["admin2"], []))
pattern = "|".join(re.escape(n) for n in all_names)

# ---- Phase 1 precision filters ----
DATELINE_RE = re.compile(r"^\*{0,2}[A-Z][a-zA-Z]+\*{0,2}\s*[-\u2013\u2014]\s*")
def strip_dateline(body):
    return DATELINE_RE.sub("", str(body), count=1)

BULLETIN_KEYWORDS = ["price bulletin", "supply chain update", "markets update", "market update"]
def is_bulletin(title):
    return any(k in str(title).lower() for k in BULLETIN_KEYWORDS)

def get_matches(row):
    if is_bulletin(row["title"]):
        return []
    text = str(row["title"]) + " " + strip_dateline(row["body"])
    return re.findall(pattern, text, flags=re.IGNORECASE)

rw["matches"] = rw.apply(get_matches, axis=1)
rw["is_matched"] = rw["matches"].apply(len) > 0

print(f"Reports matched: {rw['is_matched'].sum()} / {len(rw)}")

# ---- District-level coverage check ----
matched_districts = set()
for matches in rw[rw["is_matched"]]["matches"]:
    for m in matches:
        for _, loc in dim_location.iterrows():
            names = [loc["admin2"].lower()] + [a.lower() for a in final_alias_map.get(loc["admin2"], [])]
            if m.lower() in names:
                matched_districts.add(loc["location_id"])
                break
print(f"District-level coverage: {len(matched_districts)} / 74")

# ---- Rebuild fact_reporting using this final matching logic ----
rw["time_id"] = rw["month"].str.replace("-", "")
rw["themes"] = rw["themes"].fillna("")

months = ["202401", "202402"]
rows = []
for _, loc in dim_location.iterrows():
    for time_id in months:
        month_reports = rw[(rw["time_id"] == time_id) & (rw["is_matched"])]
        district_names = [loc["admin2"].lower()] + [a.lower() for a in final_alias_map.get(loc["admin2"], [])]
        relevant = month_reports[month_reports["matches"].apply(
            lambda matched_list: any(m.lower() in district_names for m in matched_list)
        )]
        rows.append({
            "location_id": loc["location_id"],
            "time_id": time_id,
            "reports_mentioning_district_count": len(relevant),
            "food_nutrition_report_count": relevant["themes"].str.contains("Food and Nutrition").sum(),
            "wash_report_count": relevant["themes"].str.contains("Water Sanitation Hygiene").sum(),
        })

fact_reporting = pd.DataFrame(rows)
print(f"\nfact_reporting shape: {fact_reporting.shape} (expected 74 x 2 = 148)")
fact_reporting.to_csv("fact_reporting_somalia.csv", index=False)

Reports matched: 99 / 220
District-level coverage: 50 / 74

fact_reporting shape: (148, 5) (expected 74 x 2 = 148)


# ReliefWeb Geoparsing: Validation and Improvement

## Background

`reports_mentioning_district_count` (along with its Food/Nutrition and WASH sub-counts) works by scanning each ReliefWeb report's text for district names, since ReliefWeb only tags reports at the country level. This section walks through how that matching method was checked and then improved, addressing a concern raised in supervisor feedback: the approach had never actually been checked against real report content.

## Initial Precision and Recall Assessment

Two independent random samples of 15 matched and 15 unmatched reports (30 matched, 30 unmatched total) were read by hand and classified.

**Precision** (are matched reports genuinely about the district they matched?). The first sample showed 8 of 15 (53%) genuine matches. The second showed 5 of 15 (33%). Combined, **13 of 30 matched reports (43%)** were judged genuinely about the district in question. The rest were incidental, most often national press releases naming a district only as a dateline (for example, "**Mogadishu**, The United Kingdom has donated...") or market and livestock price bulletins that list many districts purely as price comparison points, not because something happened in each one.

**Recall** (were any unmatched reports actually about a district we missed?). The first sample found no misses among its 15 unmatched reports. All were genuinely national or thematic documents, or used broad livelihood zone language instead of naming a district. The second sample found one confirmed miss: a report titled "Providing life saving support to vulnerable flood affected families in Baardheere" clearly names a real district, but that district was stored under its GADM spelling ("Baar Dheere") with no matching alias at the time, so it went uncaught.

**What this told us**: the feature reflects text mentioning a district, not a confirmed event location, which is exactly how it was named and documented from the start. Now we have an actual number behind that caution (around 43% precision) instead of an untested assumption, plus direct proof that the alias list's limited coverage (16 of 74 districts at the time) causes real, findable misses rather than just a theoretical risk.

## Phase 1: Precision Improvements

Two cheap, rule based filters were built and tested against the labeled sample before being applied to the full dataset.

1. **Dateline stripping.** A leading dateline (like "**Mogadishu**, The United Nations...") gets stripped from the report body before matching district names, so a district mentioned only in that spot no longer counts.
2. **Reference bulletin detection.** Reports whose title contains "Price Bulletin," "Supply Chain Update," or "Markets Update" are excluded entirely. A simple district count threshold was tried first and rejected, since the Livestock Price Bulletin only mentions 4 distinct districts, exactly the same as a genuine UNHCR Operational Update. Title based detection was needed instead.

Both rules were checked against 6 previously labeled cases (4 incidental, 2 genuine) and got all 6 right. Applied to the full dataset, matched reports dropped from 108 to 95, and every one of the 13 removed reports matched the reference bulletin or dateline pattern found during manual review.

**One known limitation of this fix worth flagging**: one report, "UN condemns deadly mortar attack on Aden Adde International Airport area," describes something that genuinely happened in Mogadishu, but the only mention of "Mogadishu" in the text is the dateline itself, which the rule strips. That turns a previously correct match into an incorrect miss. This is a real trade off, not a clean win. The rule helps on average but isn't error free.

## Alias List Expansion (Recall Improvement)

After confirming the Baardheere miss, the alias list was expanded in three stages, starting from an original 16 of 74 districts.

1. **Reusing crosswalk data we already had.** The 14 GADM versus common spelling differences found earlier while reconciling ACLED district names were added as aliases too, since ACLED and ReliefWeb both tend to use common spellings rather than strictly official ones. This brought coverage to 30 of 74 districts.
2. **GADM's own `VARNAME_2` field.** This field had been sitting in the GADM boundary file since it was first downloaded, just never used. Parsing it brought coverage to 65 of 74 districts, and directly confirmed the Baardheere finding: GADM's own variant list for "Baar Dheere" includes "Baardheere." Checking this field against the ACLED derived aliases also showed it isn't complete on its own. Two districts (Goldogob, Wajid) have a confirmed real variant (Galdogob, Waajid) that GADM never lists, and we only know about them because ACLED happened to use the different spelling. Because of this, both sources were merged rather than treated as interchangeable.
3. **Manual web research** on the remaining 9 districts with no alias from either source turned up genuine new variants for 2 of them. Afmadow has 6 real variants, and Hobyo has 1 ("Obbia," a historical name still used today, including as the name of Hobyo's own airport). The other 7 (Baki, Berbera, Burtinle, Eyl, Jalalaqsi, Jilib, Rab Dhuure) came back with one consistent spelling across every source checked, including operational sources like IOM's Displacement Tracking Matrix and FSNAU.

A final cleanup pass removed a few self referential entries (cases where a district's own name had accidentally been listed as its own "alias," a quirk in GADM's data), which took two districts (Doolow, Jowhar) from having a fake alias to having none. The final, corrected alias list covers **65 of 74 districts**.

## Final Combined Results

| Stage | Reports matched | Districts covered |
|---|---|---|
| Original (unvalidated) | 108 / 220 | 44 / 74 |
| Phase 1 precision filters only | 95 / 220 | 43 / 74 |
| Full alias expansion (all sources, cleaned) | **99 / 220** | **50 / 74** |

The final version matches fewer reports overall than the original (99 versus 108), and that's expected. It reflects deliberately cutting out low precision matches like reference bulletins and dateline only mentions. At the same time, district level coverage actually improved (50 versus 44), thanks to genuine matches recovered through the expanded alias list. Both changes point the same direction: a smaller but more trustworthy set of matches, covering more real districts than before.

## Remaining Limitations

- **9 districts still have no documented alias** from any source we checked (GADM, ACLED, or manual research). This should be read as "no variant found across what we checked," not proof that none exists.
- **Some genuinely ambiguous cases can't be fixed by rules alone.** One report ("Sector Commanders discuss Somalia security ahead of next stage of ATMIS drawdown") was actually classified differently by hand across the two review samples. That's a case where the right call really does depend on context, not a fixable pattern, and shows the limits of what simple rules can do.
- **The dateline stripping rule has at least one known false negative** (the Aden Adde mortar attack report), a good reminder that this is a net improvement, not a perfect fix.
- This whole review was based on a sample of 30 out of 108 originally matched reports (about 28%) plus one round of manual web research. Both should be treated as a solid, evidence backed indication rather than a fully exhaustive audit.

# Task 8: Distinguish structural availability, recorded coverage and true observability

## Structural Availability, Recorded Coverage, and True Observability

Just because every district has a row in the table doesn't mean every district is actually being observed in the same way. The coverage numbers used earlier in this project actually blend together three different questions, so this section pulls them apart for each mechanism.

**Structural availability** asks whether a source could produce a value here at all, even in theory. **Recorded coverage** asks whether the dataset actually has a value here. **True observability** asks how well that value, when it exists, reflects what's really happening on the ground. The first two can be measured directly from the data. The third mostly can't, so it's reasoned through qualitatively below.

### Conflict (ACLED)

- **Structural availability**: 100% (74/74). Media based monitoring can, in principle, pick up an event anywhere.
- **Recorded coverage**: 100% (74/74). Every district month has a value once the panel is zero filled.
- **True observability**: Hard to know. A recorded 0 could mean nothing happened, or it could mean something happened but never made it into the media ACLED tracks. Those two situations look identical in the data.

### Climate (CHIRPS/VHI)

- **Structural availability**: 100% (74/74). Satellites pass over every district the same way, regardless of what's happening on the ground.
- **Recorded coverage**: 100% (74/74). Zonal statistics get computed for every polygon without exception.
- **True observability**: Fairly strong. Since satellite measurement doesn't depend on local reporting, what's recorded is close to the real thing. One small caveat: CHIRPS is itself a modeled product blending satellite and ground gauge data, not a raw ground measurement.

### Market (WFP)

- **Structural availability**: 47% (35/74). Only districts with an actual working market being monitored can produce a price at all.
- **Recorded coverage**: 47%, minus one extra gap (Banadir in January). Nearly every district that could report a price actually does.
- **True observability**: Mixed even among the districts that are covered. Price quality depends on things like how often an enumerator visits and which vendors get checked, which isn't something this dataset can see directly.

### Reporting (ReliefWeb)

- **Structural availability**: Effectively 100%. Any district could theoretically get named in a report. There's no physical requirement like needing a market.
- **Recorded coverage**: 68% (50/74) after the geoparsing improvements, well below the structural ceiling.
- **True observability**: Weaker than the other three. Manual checking found only around 43% precision among matched reports, and a district with zero matches might still have had real events that just never got picked up by any report.

### Why this split actually matters

Market's gap and reporting's gap look similar on paper (both under 100%) but come from completely different places. Market's gap is basically forced by structural availability, there just isn't much room for the two numbers to differ. Reporting's gap isn't forced by anything physical, its structural ceiling is basically unlimited, so falling short of it means something was genuinely missed, not that it was impossible to catch in the first place. Lumping both under one "coverage" number would make two very different problems look the same, when they really call for different explanations and different caveats.

# Task 9: Present the current pilot results as exploratory rather than conclusive.

### Pilot Analysis 1: Result & Conclusion (Updated)

| Measure | Avg, with market | Avg, without market | p-value |
|---|---|---|---|
| Conflict event count | 4.57 | 3.24 | 0.18 |
| Fatalities | 4.61 | 7.37 | 0.47 |

*(`conflict_density` removed from this comparison, consistent with its removal from the feature set after testing showed it added little beyond raw event count.)*

**Result**: no significant difference on either remaining measure (both p > 0.1). The direction wasn't even consistent, a pattern more typical of noise than a real effect.

## Pilot Analysis

The two experiments below exist to show that the integrated dataset supports real cross-mechanism analysis, questions that neither source could answer sitting on its own. They are demonstrations of the dataset's usefulness, not findings to be treated as settled. With only two months of data for one country, neither result should be read as evidence of a real-world pattern, only as evidence that the dataset can support this kind of question once a larger, more capable sample exists.

### Pilot Analysis 1: Does market coverage track conflict?

**Conclusion (exploratory)**: nothing here should be read as confirming or ruling out a real relationship between market coverage and conflict. What can be said is narrower: this specific test, on this specific two-month sample, did not turn up a significant pattern. A different window, a different pair of months, or the full multi-year dataset could easily produce a different result, and this analysis should be re-run once that data exists rather than treated as a closed question.

### Pilot Analysis 2: Does reporting attention follow conflict?

**Conclusion (exploratory)**: the correlation found here, and its explanation through a one-month reporting lag, is a genuinely interesting pattern worth taking seriously, not a proven mechanism. It rests on 74 districts observed for two months, and the lag explanation itself was only possible to test because there happened to be exactly two months available, one pair to check. This result is best understood as a hypothesis worth carrying into the full study, where it can be tested against many more months and a real chance to see whether the lag pattern holds up, rather than as a conclusion about how ReliefWeb reporting behaves in general.

# Expanding the prototype

We will now expand the prototype previously built. Somalia will be kept as the reference country, but we will be using the entire 2024 year instead.

# Downloading VHI data for 2024

In [14]:
# ============================================================
# DOWNLOAD NOAA VHI DATA FOR 2024 TO GOOGLE DRIVE
# ============================================================

# ------------------------------------------------------------
# 1. Import libraries
# ------------------------------------------------------------

import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from google.colab import drive


# ------------------------------------------------------------
# 2. Mount Google Drive
# ------------------------------------------------------------

drive.mount('/content/drive')


# ------------------------------------------------------------
# 3. Set download folder
# ------------------------------------------------------------

# Change this path if you want the files stored somewhere else
DOWNLOAD_DIR = "/content/drive/MyDrive/Dissertation/Data/VHI/2024"

# Create the folder if it doesn't already exist
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

print("Files will be saved to:")
print(DOWNLOAD_DIR)


# ------------------------------------------------------------
# 4. NOAA VHI GeoTIFF directory
# ------------------------------------------------------------

BASE_URL = (
    "https://www.star.nesdis.noaa.gov/data/pub0018/"
    "VHPdata4users/data/Blended_VH_4km/geo_TIFF/"
)


# ------------------------------------------------------------
# 5. Read the NOAA directory
# ------------------------------------------------------------

print("\nConnecting to NOAA...")

response = requests.get(BASE_URL)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

print("Connection successful.")


# ------------------------------------------------------------
# 6. Find all VHI files for 2024
# ------------------------------------------------------------

vhi_files = []

for link in soup.find_all("a"):

    filename = link.get("href")

    if filename is None:
        continue

    # Keep only:
    # - files from 2024
    # - VHI files
    # - GeoTIFF files

    if (
        "P2024" in filename
        and filename.endswith(".VH.VHI.tif")
    ):
        vhi_files.append(filename)


# Remove duplicates and sort chronologically
vhi_files = sorted(set(vhi_files))


# ------------------------------------------------------------
# 7. Show files found
# ------------------------------------------------------------

print("\n----------------------------------------")
print("VHI FILES FOUND")
print("----------------------------------------")

print(f"\nTotal files found: {len(vhi_files)}\n")

for filename in vhi_files:
    print(filename)


# Stop here if no files were found
if len(vhi_files) == 0:
    raise ValueError(
        "No 2024 VHI files were found. "
        "Check the NOAA directory or filename pattern."
    )


# ------------------------------------------------------------
# 8. Download files
# ------------------------------------------------------------

print("\n----------------------------------------")
print("STARTING DOWNLOAD")
print("----------------------------------------\n")

for i, filename in enumerate(vhi_files, start=1):

    file_url = urljoin(BASE_URL, filename)

    output_path = os.path.join(
        DOWNLOAD_DIR,
        filename
    )

    # Skip files already downloaded
    if os.path.exists(output_path):

        print(
            f"[{i}/{len(vhi_files)}] "
            f"Already exists — skipping: {filename}"
        )

        continue


    print(
        f"[{i}/{len(vhi_files)}] "
        f"Downloading: {filename}"
    )


    # Download file
    with requests.get(
        file_url,
        stream=True
    ) as r:

        r.raise_for_status()

        with open(output_path, "wb") as f:

            for chunk in r.iter_content(
                chunk_size=1024 * 1024
            ):

                if chunk:
                    f.write(chunk)


# ------------------------------------------------------------
# 9. Verify downloaded files
# ------------------------------------------------------------

print("\n----------------------------------------")
print("DOWNLOAD COMPLETE")
print("----------------------------------------")

downloaded_files = sorted([
    f
    for f in os.listdir(DOWNLOAD_DIR)
    if f.endswith(".VH.VHI.tif")
])

print(
    f"\nTotal VHI files currently in "
    f"Google Drive: {len(downloaded_files)}"
)

print("\nFiles saved:\n")

for filename in downloaded_files:
    print(filename)


# ------------------------------------------------------------
# 10. Show Google Drive location
# ------------------------------------------------------------

print("\n----------------------------------------")
print("GOOGLE DRIVE LOCATION")
print("----------------------------------------")

print(DOWNLOAD_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Files will be saved to:
/content/drive/MyDrive/Dissertation/Data/VHI/2024

Connecting to NOAA...
Connection successful.

----------------------------------------
VHI FILES FOUND
----------------------------------------

Total files found: 52

VHP.G04.C07.j01.P2024001.VH.VHI.tif
VHP.G04.C07.j01.P2024002.VH.VHI.tif
VHP.G04.C07.j01.P2024003.VH.VHI.tif
VHP.G04.C07.j01.P2024004.VH.VHI.tif
VHP.G04.C07.j01.P2024005.VH.VHI.tif
VHP.G04.C07.j01.P2024006.VH.VHI.tif
VHP.G04.C07.j01.P2024007.VH.VHI.tif
VHP.G04.C07.j01.P2024008.VH.VHI.tif
VHP.G04.C07.j01.P2024009.VH.VHI.tif
VHP.G04.C07.j01.P2024010.VH.VHI.tif
VHP.G04.C07.j01.P2024011.VH.VHI.tif
VHP.G04.C07.j01.P2024012.VH.VHI.tif
VHP.G04.C07.j01.P2024013.VH.VHI.tif
VHP.G04.C07.j01.P2024014.VH.VHI.tif
VHP.G04.C07.j01.P2024015.VH.VHI.tif
VHP.G04.C07.j01.P2024016.VH.VHI.tif
VHP.G04.C07.j01.P2024017.VH.VHI.tif
VHP.G04.C07.j01.

# Downloading CHIRPS data for 2024

In [10]:
# DOWNLOAD CHIRPS MONTHLY DATA FOR 2024 TO GOOGLE DRIVE

import os
import gzip
import shutil
import requests
from google.colab import drive


# 1. Mount Google Drive

drive.mount("/content/drive")


# 2. Set folder in Google Drive

DOWNLOAD_DIR = "/content/drive/MyDrive/Dissertation/Data/CHIRPS/2024"

os.makedirs(DOWNLOAD_DIR, exist_ok=True)

print("Files will be saved to:")
print(DOWNLOAD_DIR)


# 3. Set CHIRPS monthly GeoTIFF archive

BASE_URL = (
    "https://data.chc.ucsb.edu/products/"
    "CHIRPS-2.0/global_monthly/tifs/"
)

YEAR = 2024


# 4. Create filenames for all 12 months

chirps_files = [
    f"chirps-v2.0.{YEAR}.{month:02d}.tif.gz"
    for month in range(1, 13)
]

print("\nFiles to download:\n")

for filename in chirps_files:
    print(filename)


# 5. Download and decompress each file

for i, filename in enumerate(chirps_files, start=1):

    file_url = BASE_URL + filename

    gz_path = os.path.join(
        DOWNLOAD_DIR,
        filename
    )

    tif_filename = filename.replace(".gz", "")

    tif_path = os.path.join(
        DOWNLOAD_DIR,
        tif_filename
    )

    print(f"\n[{i}/12] Processing {filename}")

    # Skip if decompressed TIFF already exists

    if os.path.exists(tif_path):

        print("TIFF already exists — skipping.")
        continue


    # Download compressed file if needed

    if not os.path.exists(gz_path):

        print("Downloading...")

        with requests.get(
            file_url,
            stream=True
        ) as r:

            r.raise_for_status()

            with open(gz_path, "wb") as f:

                for chunk in r.iter_content(
                    chunk_size=1024 * 1024
                ):

                    if chunk:
                        f.write(chunk)

    else:

        print("Compressed file already downloaded.")


    # Decompress .tif.gz to .tif

    print("Decompressing...")

    with gzip.open(gz_path, "rb") as f_in:

        with open(tif_path, "wb") as f_out:

            shutil.copyfileobj(
                f_in,
                f_out
            )


    # Delete compressed file to save Google Drive space

    os.remove(gz_path)

    print(f"Saved: {tif_filename}")


# 6. Verify downloaded files

downloaded_files = sorted([
    f
    for f in os.listdir(DOWNLOAD_DIR)
    if f.endswith(".tif")
])

print("\nDOWNLOAD COMPLETE")

print(
    f"\nTotal CHIRPS TIFF files found: "
    f"{len(downloaded_files)}"
)

print("\nFiles:\n")

for filename in downloaded_files:
    print(filename)

print("\nGoogle Drive folder:")
print(DOWNLOAD_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Files will be saved to:
/content/drive/MyDrive/Dissertation/Data/CHIRPS/2024

Files to download:

chirps-v2.0.2024.01.tif.gz
chirps-v2.0.2024.02.tif.gz
chirps-v2.0.2024.03.tif.gz
chirps-v2.0.2024.04.tif.gz
chirps-v2.0.2024.05.tif.gz
chirps-v2.0.2024.06.tif.gz
chirps-v2.0.2024.07.tif.gz
chirps-v2.0.2024.08.tif.gz
chirps-v2.0.2024.09.tif.gz
chirps-v2.0.2024.10.tif.gz
chirps-v2.0.2024.11.tif.gz
chirps-v2.0.2024.12.tif.gz

[1/12] Processing chirps-v2.0.2024.01.tif.gz
Downloading...
Decompressing...
Saved: chirps-v2.0.2024.01.tif

[2/12] Processing chirps-v2.0.2024.02.tif.gz
Downloading...
Decompressing...
Saved: chirps-v2.0.2024.02.tif

[3/12] Processing chirps-v2.0.2024.03.tif.gz
Downloading...
Decompressing...
Saved: chirps-v2.0.2024.03.tif

[4/12] Processing chirps-v2.0.2024.04.tif.gz
Downloading...
Decompressing...
Saved: chirps-v2.0.2024.04.tif

[5/12] Proce

# Downloading ReliefWeb data for 2024

In [13]:
# DOWNLOAD RELIEFWEB REPORTS FOR 2024 TO GOOGLE DRIVE

import os
import requests
import pandas as pd
from google.colab import drive


# 1. Mount Google Drive

drive.mount("/content/drive")


# 2. Set output folder

DOWNLOAD_DIR = "/content/drive/MyDrive/Dissertation/Data/ReliefWeb/2024"

os.makedirs(DOWNLOAD_DIR, exist_ok=True)

OUTPUT_FILE = os.path.join(
    DOWNLOAD_DIR,
    "reliefweb_horn_of_africa_2024.csv"
)

print("File will be saved to:")
print(OUTPUT_FILE)


# 3. Enter your approved ReliefWeb app name

APP_NAME = "MiddlesexUniversity-FoodSecurityResearch-X7K9"


# 4. Set ReliefWeb API endpoint

BASE_URL = (
    f"https://api.reliefweb.int/v2/reports"
    f"?appname={APP_NAME}"
)


# 5. Define countries and date range

COUNTRIES = [
    "Somalia",
    "Ethiopia",
    "Kenya",
    "South Sudan"
]

START_DATE = "2024-01-01T00:00:00+00:00"
END_DATE = "2024-12-31T23:59:59+00:00"


# 6. Define fields to download

FIELDS_TO_INCLUDE = [
    "id",
    "title",
    "body",
    "url",
    "date.created",
    "date.original",
    "date.changed",
    "primary_country",
    "country",
    "source",
    "format",
    "theme",
    "disaster",
    "disaster_type",
    "language",
    "headline",
    "status"
]


# 7. Create helper functions

def get_name_list(items):
    if not items:
        return None

    return ", ".join(
        item.get("name", "")
        for item in items
        if isinstance(item, dict)
    )


def get_shortname_list(items):
    if not items:
        return None

    return ", ".join(
        item.get("shortname", item.get("name", ""))
        for item in items
        if isinstance(item, dict)
    )


def flatten_report(item):

    fields = item.get("fields", {})

    date = fields.get("date", {})

    primary_country = fields.get(
        "primary_country",
        {}
    )

    return {
        "id": item.get("id"),

        "title": fields.get("title"),

        "body": fields.get("body"),

        "url": fields.get("url"),

        "date_created": date.get("created"),

        "date_original": date.get("original"),

        "date_changed": date.get("changed"),

        "primary_country": (
            primary_country.get("name")
            if isinstance(primary_country, dict)
            else None
        ),

        "countries": get_name_list(
            fields.get("country")
        ),

        "sources": get_name_list(
            fields.get("source")
        ),

        "source_shortnames": get_shortname_list(
            fields.get("source")
        ),

        "formats": get_name_list(
            fields.get("format")
        ),

        "themes": get_name_list(
            fields.get("theme")
        ),

        "disasters": get_name_list(
            fields.get("disaster")
        ),

        "disaster_types": get_name_list(
            fields.get("disaster_type")
        ),

        "languages": get_name_list(
            fields.get("language")
        ),

        "headline": fields.get("headline"),

        "status": fields.get("status")
    }


# 8. Create function to download one country using pagination

def download_country_reports(country):

    print(f"\nDownloading ReliefWeb reports for {country}...")

    all_reports = []

    offset = 0
    limit = 1000

    while True:

        payload = {
            "limit": limit,

            "offset": offset,

            "profile": "full",

            "sort": [
                "date.created:asc"
            ],

            "filter": {
                "operator": "AND",

                "conditions": [

                    {
                        "field": "primary_country",
                        "value": country
                    },

                    {
                        "field": "date.created",

                        "value": {
                            "from": START_DATE,
                            "to": END_DATE
                        }
                    }

                ]
            },

            "fields": {
                "include": FIELDS_TO_INCLUDE
            }
        }

        response = requests.post(
            BASE_URL,
            json=payload
        )

        response.raise_for_status()

        data = response.json()

        reports = data.get(
            "data",
            []
        )

        total_count = data.get(
            "totalCount",
            0
        )

        print(
            f"Retrieved "
            f"{offset + len(reports):,} "
            f"of {total_count:,}"
        )

        all_reports.extend(reports)

        if (
            len(reports) == 0
            or offset + len(reports) >= total_count
        ):
            break

        offset += limit


    flattened = [
        flatten_report(report)
        for report in all_reports
    ]

    country_df = pd.DataFrame(
        flattened
    )

    print(
        f"{country}: "
        f"{len(country_df):,} reports downloaded"
    )

    return country_df


# 9. Download all four countries

all_country_data = []

for country in COUNTRIES:

    country_df = download_country_reports(
        country
    )

    all_country_data.append(
        country_df
    )


# 10. Combine all countries

reliefweb_2024 = pd.concat(
    all_country_data,
    ignore_index=True
)

print(
    "\nTotal reports downloaded:",
    f"{len(reliefweb_2024):,}"
)


# 11. Convert dates to datetime

date_columns = [
    "date_created",
    "date_original",
    "date_changed"
]

for column in date_columns:

    reliefweb_2024[column] = pd.to_datetime(
        reliefweb_2024[column],
        errors="coerce",
        utc=True
    )


# 12. Create year and month columns

reliefweb_2024["year"] = (
    reliefweb_2024["date_created"]
    .dt.year
)

reliefweb_2024["month"] = (
    reliefweb_2024["date_created"]
    .dt.month
)


# 13. Check country and monthly coverage

print("\nReports by primary country:")

print(
    reliefweb_2024[
        "primary_country"
    ].value_counts(
        dropna=False
    )
)

print("\nReports by month:")

print(
    reliefweb_2024[
        ["year", "month"]
    ]
    .value_counts()
    .sort_index()
)


# 14. Check duplicate report IDs

print(
    "\nDuplicate report IDs:",
    reliefweb_2024[
        "id"
    ].duplicated().sum()
)


# 15. Remove duplicate reports if necessary

reliefweb_2024 = (
    reliefweb_2024
    .drop_duplicates(
        subset="id"
    )
    .reset_index(drop=True)
)


# 16. Save dataset to Google Drive

reliefweb_2024.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\nDownload complete.")

print(
    "Final number of unique reports:",
    f"{len(reliefweb_2024):,}"
)

print("\nSaved to:")
print(OUTPUT_FILE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
File will be saved to:
/content/drive/MyDrive/Dissertation/Data/ReliefWeb/2024/reliefweb_horn_of_africa_2024.csv

Retrieved 1,000 of 1,336
Retrieved 1,336 of 1,336
Somalia: 1,336 reports downloaded

Retrieved 847 of 847
Ethiopia: 847 reports downloaded

Retrieved 594 of 594
Kenya: 594 reports downloaded

Retrieved 910 of 910
South Sudan: 910 reports downloaded

Total reports downloaded: 3,687

Reports by primary country:
primary_country
Somalia        1336
South Sudan     910
Ethiopia        847
Kenya           594
Name: count, dtype: int64

Reports by month:
year  month
2024  1        273
      2        324
      3        328
      4        298
      5        331
      6        297
      7        366
      8        308
      9        266
      10       286
      11       339
      12       271
Name: count, dtype: int64

Duplicate report IDs: 0

Download comp